In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score

import random
import os



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =========================
# LOAD N-BACK
# =========================
data_nb = torch.load(
    "/content/drive/MyDrive/eeg_preprocessed_nback.pt",
    weights_only=False
)

X_time_t = data_nb["X_time"]
X_bp_t   = data_nb["X_bp"]
X_cov_t  = data_nb["X_cov"]
X_plv_t  = data_nb["X_plv"]
y_t      = data_nb["y"]
subjects = data_nb["subjects"]

print("Loaded N-back")


# =========================
# LOAD HTC
# =========================
data_htc = torch.load(
    "/content/drive/MyDrive/eeg_preprocessed_htc.pt",
    weights_only=False
)

X_time_t_htc = data_htc["X_time"]
X_bp_t_htc   = data_htc["X_bp"]
X_cov_t_htc  = data_htc["X_cov"]
X_plv_t_htc  = data_htc["X_plv"]
y_t_htc      = data_htc["y"]
subjects_htc = data_htc["subjects"]

print("Loaded HTC")

Loaded N-back
Loaded HTC


In [ ]:
print("\n--- N-BACK ---")
print(X_time_t.shape, X_bp_t.shape, X_cov_t.shape, X_plv_t.shape, y_t.shape)

print("\n--- HTC ---")
print(X_time_t_htc.shape, X_bp_t_htc.shape, X_cov_t_htc.shape, X_plv_t_htc.shape, y_t_htc.shape)


--- N-BACK ---
torch.Size([12159, 14, 512]) torch.Size([12159, 42]) torch.Size([15007, 105]) torch.Size([15007, 210]) torch.Size([12159])

--- HTC ---
torch.Size([5056, 14, 512]) torch.Size([5056, 42]) torch.Size([5056, 105]) torch.Size([5056, 210]) torch.Size([5056])


In [ ]:
def extract_embeddings(model, loader, device):
    model.eval()
    E = []
    Y = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            db      = db.to(device)

            # SAFE: supports both versions of model
            try:
                e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi, db)
            except TypeError:
                e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            E.append(e.cpu())
            Y.append(yb.cpu())

    return torch.cat(E, dim=0), torch.cat(Y, dim=0)

In [ ]:
import numpy as np

# ----------------------------
# REMOVE 4 LOWEST HTC SUBJECTS
# ----------------------------
htc_perf = {
    "subject_01": 0.7677902579307556,
    "subject_02": 0.8880308866500854,
    "subject_03": 0.7392995953559875,
    "subject_06": 0.7952755689620972,
    "subject_08": 0.6611570119857788,
    "subject_12": 0.8793774247169495,
    "subject_16": 0.7346153855323792,
    "subject_17": 0.73046875,
    "subject_18": 0.8046875,
    "subject_19": 0.957198441028595,
    "subject_20": 0.6640926599502563,
    "subject_21": 0.8560311198234558,
    "subject_22": 0.8615384697914124,
    "subject_23": 0.8759689927101135,
    "subject_24": 0.9372549653053284,
    "subject_25": 0.6100385785102844,
    "subject_26": 0.8859315514564514,
}

K = 4
sorted_subjs = sorted(htc_perf.items(), key=lambda x: x[1])
remove_subjs = [s for s, _ in sorted_subjs[:K]]

print("Removing HTC subjects:", remove_subjs)

keep_mask = np.array([s not in remove_subjs for s in subjects_htc])

X_time_t_htc = X_time_t_htc[keep_mask]
X_bp_t_htc   = X_bp_t_htc[keep_mask]
X_cov_t_htc  = X_cov_t_htc[keep_mask]
X_plv_t_htc  = X_plv_t_htc[keep_mask]
y_t_htc      = y_t_htc[keep_mask]
subjects_htc = subjects_htc[keep_mask]

all_subjects_htc = np.unique(subjects_htc)
print("Remaining HTC subjects:", len(all_subjects_htc), all_subjects_htc)

Removing HTC subjects: ['subject_25', 'subject_08', 'subject_20', 'subject_17']
Remaining HTC subjects: 13 ['subject_01' 'subject_02' 'subject_03' 'subject_06' 'subject_12'
 'subject_16' 'subject_18' 'subject_19' 'subject_21' 'subject_22'
 'subject_23' 'subject_24' 'subject_26']


In [ ]:
all_subjects = np.unique(subjects)
print("\nSubjects list:", all_subjects)


Subjects list: ['subject_01' 'subject_02' 'subject_03' 'subject_04' 'subject_06'
 'subject_07' 'subject_08' 'subject_09' 'subject_10' 'subject_12'
 'subject_13' 'subject_14' 'subject_15']


In [ ]:
HTC_ID = 0
NBACK_ID = 1

subjects_nb_prefixed  = np.array([f"nb_{s}" for s in subjects])
subjects_htc_prefixed = np.array([f"htc_{s}" for s in subjects_htc])

print("Example N-back subjects:", subjects_nb_prefixed[:5])
print("Example HTC subjects:", subjects_htc_prefixed[:5])

Example N-back subjects: ['nb_subject_01' 'nb_subject_01' 'nb_subject_01' 'nb_subject_01'
 'nb_subject_01']
Example HTC subjects: ['htc_subject_01' 'htc_subject_01' 'htc_subject_01' 'htc_subject_01'
 'htc_subject_01']


In [ ]:
import torch
from torch.utils.data import Dataset

class EEGDataset(Dataset):
    def __init__(self, X_time, X_bp, X_cov, X_plv, X_roi, y, subj_strs, dataset_id):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_cov  = X_cov
        self.X_plv  = X_plv
        self.X_roi  = X_roi
        self.y      = y

        self.dataset_id = int(dataset_id)

        uniq = np.unique(subj_strs)
        self.subj_map = {s: i for i, s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.X_bp[idx],
            self.X_cov[idx],
            self.X_plv[idx],
            self.X_roi[idx],
            self.y[idx],
            self.subj[idx],
            torch.tensor(self.dataset_id, dtype=torch.long)
        )

In [ ]:
import torch.nn.functional as F

def supcon_within_dataset_loss(proj, y, subj, dataset, temperature=0.1):
    proj = F.normalize(proj, dim=1)
    B = proj.shape[0]

    sim = (proj @ proj.T) / temperature
    exp_sim = torch.exp(sim)

    y = y.unsqueeze(0)
    subj = subj.unsqueeze(0)
    dataset = dataset.unsqueeze(0)

    same_label   = (y == y.T)
    diff_subj    = (subj != subj.T)
    same_dataset = (dataset == dataset.T)

    positives = same_label & diff_subj & same_dataset

    eye = torch.eye(B, dtype=torch.bool, device=proj.device)
    positives = positives & ~eye
    valid = ~eye

    pos_exp = exp_sim * positives
    all_exp = exp_sim * valid

    pos_sum = pos_exp.sum(dim=1)
    denom   = all_exp.sum(dim=1) + 1e-8

    has_pos = positives.sum(dim=1) > 0
    if has_pos.sum() == 0:
        return torch.tensor(0.0, device=proj.device)

    log_prob = torch.log((pos_sum[has_pos] + 1e-8) / denom[has_pos])
    loss = -log_prob.mean()
    return loss

In [ ]:
from torch.utils.data import Sampler
import random
from collections import defaultdict

class JointBalancedBatchSampler(Sampler):
    def __init__(self, dataset, batch_size, subjects_per_dataset=6):
        self.dataset = dataset
        self.batch_size = batch_size
        self.subjects_per_dataset = subjects_per_dataset

        self.group_to_indices = defaultdict(list)

        for idx in range(len(dataset)):
            _, _, _, _, _, _, subj, db = dataset[idx]
            key = (int(db.item()), int(subj.item()))
            self.group_to_indices[key].append(idx)

        self.nb_groups  = [g for g in self.group_to_indices if g[0] == NBACK_ID]
        self.htc_groups = [g for g in self.group_to_indices if g[0] == HTC_ID]

        assert len(self.nb_groups) > 0, "No N-back groups found"
        assert len(self.htc_groups) > 0, "No HTC groups found"

        self.nb_total  = sum(len(self.group_to_indices[g]) for g in self.nb_groups)
        self.htc_total = sum(len(self.group_to_indices[g]) for g in self.htc_groups)

        self.max_samples = max(self.nb_total, self.htc_total)
        self.num_batches = self.max_samples // batch_size

        self.samples_per_subject = batch_size // (2 * subjects_per_dataset)

    def __iter__(self):
        group_lists = {}
        for g in self.group_to_indices:
            lst = self.group_to_indices[g][:]
            random.shuffle(lst)
            group_lists[g] = lst

        group_ptrs = {g: 0 for g in group_lists}

        for _ in range(self.num_batches):
            chosen_nb  = random.sample(self.nb_groups,  self.subjects_per_dataset)
            chosen_htc = random.sample(self.htc_groups, self.subjects_per_dataset)

            batch = []

            for g in chosen_nb + chosen_htc:
                for _ in range(self.samples_per_subject):
                    if group_ptrs[g] >= len(group_lists[g]):
                        random.shuffle(group_lists[g])
                        group_ptrs[g] = 0

                    batch.append(group_lists[g][group_ptrs[g]])
                    group_ptrs[g] += 1

            yield batch

    def __len__(self):
        return self.num_batches

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FusionEncoder(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=42, roi_dim=3):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim   # 42 + 3 = 45

        # ------------------
        # TIME BRANCH
        # ------------------
        self.time_branch = nn.Sequential(
            nn.Conv1d(14, 14, kernel_size=7, padding=3, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 14, kernel_size=15, padding=7, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        # ------------------
        # BANDPOWER + ROI BRANCH
        # ------------------
        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),   # was 42 → now 45
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        # ------------------
        # COVARIANCE BRANCH
        # ------------------
        self.cov_branch = nn.Sequential(
            nn.Linear(105, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32)
        )

        # ------------------
        # PLV BRANCH
        # ------------------
        self.plv_branch = nn.Sequential(
            nn.Linear(210, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32)
        )

        # ------------------
        # FUSION
        # ------------------
        self.embed = nn.Sequential(
            nn.Linear(32 + 32 + 32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_cov, x_plv, x_roi):

        # ------------------
        # Time branch
        # ------------------
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)  # (B,32)

        # ------------------
        # Bandpower + ROI
        # ------------------
        bp_in = torch.cat([x_bp, x_roi], dim=1)  # (B,45)
        z_bp = self.bp_branch(bp_in)

        # ------------------
        # Covariance
        # ------------------
        z_cov = self.cov_branch(x_cov)

        # ------------------
        # PLV
        # ------------------
        z_plv = self.plv_branch(x_plv)

        # ------------------
        # Fusion
        # ------------------
        z = torch.cat([z_time, z_bp, z_cov, z_plv], dim=1)
        e = self.embed(z)

        return e

In [ ]:
import torch.nn as nn

class ContrastiveModel(nn.Module):
    def __init__(self,
                 emb_dim=64,
                 proj_dim=64,
                 bp_dim=42,
                 roi_dim=3):

        super().__init__()

        # Main encoder (learns embedding geometry)
        self.encoder = FusionEncoder(
            emb_dim=emb_dim,
            bp_dim=bp_dim,
            roi_dim=roi_dim
        )

        # Projection head for SupCon loss
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_cov, x_plv, x_roi):
        # Embedding
        e = self.encoder(x_time, x_bp, x_cov, x_plv, x_roi)

        # Projection (used only for contrastive loss)
        p = self.proj(e)

        return e, p

In [ ]:
def prototype_loss(emb, y):
    """
    emb: (B, D) embedding vectors (from encoder)
    y:   (B,) class labels
    """

    B, D = emb.shape
    loss = 0.0
    count = 0

    classes = torch.unique(y)

    for c in classes:
        mask = (y == c)
        if mask.sum() < 2:
            continue

        class_emb = emb[mask]              # (Nc, D)
        proto = class_emb.mean(dim=0)      # (D,)

        loss += ((class_emb - proto)**2).sum(dim=1).mean()
        count += 1

    if count > 0:
        return loss / count
    else:
        return torch.tensor(0.0, device=emb.device)

In [ ]:
import torch.nn as nn

def set_partial_bn_adapt(model: nn.Module, allow=("time_branch",)):
    """
    Put BN layers in *specified submodules* into train mode (update running stats).
    Keep all other BN in eval mode.
    Always keep Dropout in eval mode.
    """
    # Default everything to eval
    model.eval()

    # Dropout always eval
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.eval()

    # Enable BN train only in allowed named submodules
    for name, module in model.named_modules():
        if any(name.startswith(a) or (f".{a}." in name) or name.endswith(a) for a in allow):
            for m in module.modules():
                if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                    m.train()

In [ ]:
import torch.nn.functional as F

def prototype_ce_loss(e, y, temperature=0.1):
    """
    e: (B, D) embeddings
    y: (B,) labels
    """

    classes = torch.unique(y)
    prototypes = []

    for c in classes:
        prototypes.append(e[y == c].mean(dim=0))

    prototypes = torch.stack(prototypes)  # (C, D)

    # normalize for cosine
    e_norm = F.normalize(e, dim=1)
    proto_norm = F.normalize(prototypes, dim=1)

    logits = e_norm @ proto_norm.T  # (B, C)
    logits = logits / temperature

    # map labels to [0, C-1]
    label_map = {c.item(): i for i, c in enumerate(classes)}
    y_mapped = torch.tensor([label_map[int(lbl.item())] for lbl in y],
                            device=y.device)

    loss = F.cross_entropy(logits, y_mapped)
    return loss

In [ ]:

import torch
import torch.nn.functional as F

def mixed_dataset_metric_loss(
    e, proj, yb, sb, db,
    lambda_con=0.5,
    proto_lambda=0.10,
    temperature=0.1
):
    # ----------------------------
    # normalize embeddings
    # ----------------------------
    e = F.normalize(e, dim=1)
    proj = F.normalize(proj, dim=1)

    loss_metric = torch.tensor(0.0, device=e.device)
    loss_proto  = torch.tensor(0.0, device=e.device)

    mask_nb  = (db == NBACK_ID)
    mask_htc = (db == HTC_ID)

    # ----------------------------
    # PROTOTYPE + METRIC
    # ----------------------------
    if mask_nb.any():
        loss_metric += prototype_ce_loss(
            e[mask_nb], yb[mask_nb], temperature=temperature
        )
        loss_proto += prototype_loss(
            e[mask_nb], yb[mask_nb]
        )

    if mask_htc.any():
        loss_metric += prototype_ce_loss(
            e[mask_htc], yb[mask_htc], temperature=temperature
        )
        loss_proto += prototype_loss(
            e[mask_htc], yb[mask_htc]
        )

    # ----------------------------
    # SUPCON
    # ----------------------------
    loss_con = supcon_within_dataset_loss(
        proj, yb, sb, db, temperature=temperature
    )

    # ----------------------------
    # 🔥 FIXED: DISTANCE CONSISTENCY
    # ----------------------------
    loss_dist = torch.tensor(0.0, device=e.device)

    if mask_nb.any() and mask_htc.any():

        def get_dist_vector(e_sub, y_sub):
            classes = torch.unique(y_sub)

            if len(classes) < 2:
                return None

            protos = torch.stack([
                e_sub[y_sub == c].mean(dim=0)
                for c in classes
            ])

            protos = F.normalize(protos, dim=1)

            # pairwise distances
            dists = torch.cdist(protos, protos, p=2)

            # 🔥 take upper triangle only (ignore size mismatch)
            idx = torch.triu_indices(dists.size(0), dists.size(1), offset=1)
            vec = dists[idx[0], idx[1]]

            return vec

        d_nb  = get_dist_vector(e[mask_nb], yb[mask_nb])
        d_htc = get_dist_vector(e[mask_htc], yb[mask_htc])

        if d_nb is not None and d_htc is not None:

            # normalize
            d_nb  = d_nb  / (d_nb.mean()  + 1e-6)
            d_htc = d_htc / (d_htc.mean() + 1e-6)

            # 🔥 match only overlapping length
            L = min(len(d_nb), len(d_htc))

            if L > 0:
                loss_dist = F.mse_loss(d_nb[:L], d_htc[:L])

    # ----------------------------
    # FINAL LOSS
    # ----------------------------
    total = (
        loss_metric
        + lambda_con * loss_con
        + proto_lambda * loss_proto
        + 0.1 * loss_dist
    )

    return total, loss_metric, loss_con, loss_proto

In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4
SHIFT_THRESH = 0.15

N_FOLDS = min(len(all_subjects), len(all_subjects_htc))
overall_results_joint = []

for fold_i in range(N_FOLDS):

    test_subj_nb  = all_subjects[fold_i]
    test_subj_htc = all_subjects_htc[fold_i]

    print("\n" + "="*90)
    print(f"JOINT FOLD {fold_i+1}/{N_FOLDS}")
    print(f"N-back test subject: {test_subj_nb}")
    print(f"HTC test subject:    {test_subj_htc}")
    print("="*90)

    # =========================
    # SPLIT
    # =========================
    test_mask_nb  = (subjects == test_subj_nb)
    train_mask_nb = ~test_mask_nb
    train_idx_nb  = np.where(train_mask_nb)[0]
    test_idx_nb   = np.where(test_mask_nb)[0]

    test_mask_htc  = (subjects_htc == test_subj_htc)
    train_mask_htc = ~test_mask_htc
    train_idx_htc  = np.where(train_mask_htc)[0]
    test_idx_htc   = np.where(test_mask_htc)[0]

    # =========================
    # N-BACK FEATURE PREP
    # =========================
    X_cov_train_nb = torch.zeros_like(X_cov_t[train_idx_nb])
    X_cov_test_nb  = torch.zeros_like(X_cov_t[test_idx_nb])

    X_plv_train_nb = X_plv_t[train_idx_nb].clone()
    X_plv_test_nb  = X_plv_t[test_idx_nb].clone()

    X_time_train_nb = X_time_t[train_idx_nb].clone()
    X_time_test_nb  = X_time_t[test_idx_nb].clone()

    X_bp_train_nb = X_bp_t[train_idx_nb].clone()
    X_bp_test_nb  = X_bp_t[test_idx_nb].clone()

    y_train_nb = y_t[train_idx_nb].clone()
    y_test_nb  = y_t[test_idx_nb].clone()

    time_mu_nb = X_time_train_nb.mean(dim=(0, 2), keepdim=True)
    time_sd_nb = X_time_train_nb.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_nb = (X_time_train_nb - time_mu_nb) / time_sd_nb
    X_time_test_nb  = (X_time_test_nb  - time_mu_nb) / time_sd_nb

    bp_mu_nb = X_bp_train_nb.mean(dim=0, keepdim=True)
    bp_sd_nb = X_bp_train_nb.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_nb = (X_bp_train_nb - bp_mu_nb) / bp_sd_nb
    X_bp_test_nb  = (X_bp_test_nb  - bp_mu_nb) / bp_sd_nb

    X_roi_train_nb = torch.zeros((X_bp_train_nb.shape[0], 0))
    X_roi_test_nb  = torch.zeros((X_bp_test_nb.shape[0], 0))

    # =========================
    # HTC FEATURE PREP
    # =========================
    X_time_train_htc = X_time_t_htc[train_idx_htc].clone()
    X_time_test_htc  = X_time_t_htc[test_idx_htc].clone()

    X_bp_train_htc = X_bp_t_htc[train_idx_htc].clone()
    X_bp_test_htc  = X_bp_t_htc[test_idx_htc].clone()

    X_cov_train_htc = X_cov_t_htc[train_idx_htc].clone()
    X_cov_test_htc  = X_cov_t_htc[test_idx_htc].clone()

    X_plv_train_htc = X_plv_t_htc[train_idx_htc].clone()
    X_plv_test_htc  = X_plv_t_htc[test_idx_htc].clone()

    y_train_htc = y_t_htc[train_idx_htc].clone()
    y_test_htc  = y_t_htc[test_idx_htc].clone()

    time_mu_htc = X_time_train_htc.mean(dim=(0, 2), keepdim=True)
    time_sd_htc = X_time_train_htc.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_htc = (X_time_train_htc - time_mu_htc) / time_sd_htc
    X_time_test_htc  = (X_time_test_htc  - time_mu_htc) / time_sd_htc

    bp_mu_htc = X_bp_train_htc.mean(dim=0, keepdim=True)
    bp_sd_htc = X_bp_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_htc = (X_bp_train_htc - bp_mu_htc) / bp_sd_htc
    X_bp_test_htc  = (X_bp_test_htc  - bp_mu_htc) / bp_sd_htc

    cov_mu_htc = X_cov_train_htc.mean(dim=0, keepdim=True)
    cov_sd_htc = X_cov_train_htc.std(dim=0, keepdim=True) + 1e-6
    X_cov_train_htc = (X_cov_train_htc - cov_mu_htc) / cov_sd_htc
    X_cov_test_htc  = (X_cov_test_htc  - cov_mu_htc) / cov_sd_htc

    X_roi_train_htc = torch.zeros((X_bp_train_htc.shape[0], 0))
    X_roi_test_htc  = torch.zeros((X_bp_test_htc.shape[0], 0))

    # =========================
    # DATASETS
    # =========================
    train_ds_nb = EEGDataset(
        X_time_train_nb, X_bp_train_nb, X_cov_train_nb, X_plv_train_nb, X_roi_train_nb,
        y_train_nb,
        subjects_nb_prefixed[train_idx_nb],
        dataset_id=NBACK_ID
    )

    test_ds_nb = EEGDataset(
        X_time_test_nb, X_bp_test_nb, X_cov_test_nb, X_plv_test_nb, X_roi_test_nb,
        y_test_nb,
        subjects_nb_prefixed[test_idx_nb],
        dataset_id=NBACK_ID
    )

    train_ds_htc = EEGDataset(
        X_time_train_htc, X_bp_train_htc, X_cov_train_htc, X_plv_train_htc, X_roi_train_htc,
        y_train_htc,
        subjects_htc_prefixed[train_idx_htc],
        dataset_id=HTC_ID
    )

    test_ds_htc = EEGDataset(
        X_time_test_htc, X_bp_test_htc, X_cov_test_htc, X_plv_test_htc, X_roi_test_htc,
        y_test_htc,
        subjects_htc_prefixed[test_idx_htc],
        dataset_id=HTC_ID
    )

    train_ds_joint = ConcatDataset([train_ds_nb, train_ds_htc])

    sampler = JointBalancedBatchSampler(
        train_ds_joint,
        batch_size=BATCH,
        subjects_per_dataset=6
    )

    train_loader         = DataLoader(train_ds_joint, batch_sampler=sampler)
    train_eval_loader_nb = DataLoader(train_ds_nb, batch_size=256, shuffle=False)
    train_eval_loader_htc = DataLoader(train_ds_htc, batch_size=256, shuffle=False)
    test_loader_nb       = DataLoader(test_ds_nb, batch_size=256, shuffle=False)
    test_loader_htc      = DataLoader(test_ds_htc, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_score = -1.0
    best_state = None
    best_acc_nb = -1.0
    best_acc_htc = -1.0

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)
            db      = db.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss, loss_metric, loss_con, loss_proto = mixed_dataset_metric_loss(
                e, proj, yb, sb, db,
                lambda_con=LAMBDA,
                proto_lambda=PROTO_LAMBDA,
                temperature=TEMP
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL: NB
        # =========================
        model.eval()
        with torch.no_grad():
            e_train_nb, y_train_nb_full = extract_embeddings(model, train_eval_loader_nb, device)
            e_test_nb,  y_test_nb_full  = extract_embeddings(model, test_loader_nb, device)

            classes_nb = torch.unique(y_train_nb_full)
            protos_nb = torch.stack([
                e_train_nb[y_train_nb_full == c].mean(dim=0) for c in classes_nb
            ])

            logits_nb = F.normalize(e_test_nb, dim=1) @ F.normalize(protos_nb, dim=1).T
            pred_nb = logits_nb.argmax(dim=1)

            label_map_nb = {int(c.item()): i for i, c in enumerate(classes_nb)}
            y_test_nb_local = torch.tensor(
                [label_map_nb[int(v.item())] for v in y_test_nb_full],
                device=pred_nb.device
            )

            acc_nb = (pred_nb == y_test_nb_local).float().mean().item()

            # =========================
            # ZERO-SHOT EVAL: HTC
            # =========================
            e_train_htc, y_train_htc_full = extract_embeddings(model, train_eval_loader_htc, device)
            e_test_htc,  y_test_htc_full  = extract_embeddings(model, test_loader_htc, device)

            classes_htc = torch.unique(y_train_htc_full)
            protos_htc = torch.stack([
                e_train_htc[y_train_htc_full == c].mean(dim=0) for c in classes_htc
            ])

            logits_htc = F.normalize(e_test_htc, dim=1) @ F.normalize(protos_htc, dim=1).T
            pred_htc = logits_htc.argmax(dim=1)

            label_map_htc = {int(c.item()): i for i, c in enumerate(classes_htc)}
            y_test_htc_local = torch.tensor(
                [label_map_htc[int(v.item())] for v in y_test_htc_full],
                device=pred_htc.device
            )

            acc_htc = (pred_htc == y_test_htc_local).float().mean().item()

        score = 0.5 * acc_nb + 0.5 * acc_htc

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            best_acc_nb = acc_nb
            best_acc_htc = acc_htc

        print(f"Epoch {ep:02d} | NB zero-shot {acc_nb:.4f} | HTC zero-shot {acc_htc:.4f} | avg {score:.4f}")

    # =====================================================
    # LOAD BEST BASE MODEL
    # =====================================================
    model.load_state_dict(best_state)
    best_joint_state = copy.deepcopy(best_state)

    # =====================================================
    # SHIFT COMPUTATION ON BEST BASE MODEL
    # =====================================================
    model.eval()
    with torch.no_grad():
        e_train_nb_base, y_train_nb_base = extract_embeddings(model, train_eval_loader_nb, device)
        e_test_nb_base,  y_test_nb_base  = extract_embeddings(model, test_loader_nb, device)

        e_train_htc_base, y_train_htc_base = extract_embeddings(model, train_eval_loader_htc, device)
        e_test_htc_base,  y_test_htc_base  = extract_embeddings(model, test_loader_htc, device)

    shift_nb = torch.norm(e_train_nb_base.mean(0) - e_test_nb_base.mean(0)).item()
    shift_htc = torch.norm(e_train_htc_base.mean(0) - e_test_htc_base.mean(0)).item()

    print(f"Best-model shift | NB: {shift_nb:.4f} | HTC: {shift_htc:.4f}")

    # =====================================================
    # FEW-SHOT: N-BACK (GATED AdaBN + Mahalanobis)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    # start NB adaptation from the same best joint model
    model.load_state_dict(best_joint_state)
    model.eval()

    all_e_nb = e_test_nb_base.clone()
    all_y_nb = y_test_nb_base.clone()

    support_idx_nb = []
    for c in torch.unique(all_y_nb):
        class_idx = (all_y_nb == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx_nb.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx_nb = torch.cat(support_idx_nb)

    mask_nb = torch.zeros(len(all_y_nb), dtype=torch.bool)
    mask_nb[support_idx_nb] = True

    support_subset_nb = torch.utils.data.Subset(test_ds_nb, support_idx_nb.tolist())
    support_loader_nb = DataLoader(support_subset_nb, batch_size=256, shuffle=False)

    use_bn_nb = shift_nb > SHIFT_THRESH
    print(f"N-back BN USED: {use_bn_nb}")

    if use_bn_nb:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader_nb:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract test embeddings after BN adaptation
        all_e_nb, all_y_nb = extract_embeddings(model, test_loader_nb, device)

    e_support_nb = all_e_nb[mask_nb].to(device)
    y_support_nb = all_y_nb[mask_nb].to(device)
    e_query_nb   = all_e_nb[~mask_nb].to(device)
    y_query_nb   = all_y_nb[~mask_nb].to(device)

    e_support_nb = F.normalize(e_support_nb, dim=1)
    e_query_nb   = F.normalize(e_query_nb, dim=1)

    var_nb = e_support_nb.var(dim=0)
    weights_nb = 1.0 / (var_nb + 1e-6)
    weights_nb = weights_nb / weights_nb.mean()
    e_support_nb = e_support_nb * weights_nb
    e_query_nb   = e_query_nb * weights_nb

    classes_nb = torch.unique(y_support_nb)
    protos_nb = torch.stack([
        e_support_nb[y_support_nb == c].mean(dim=0) for c in classes_nb
    ])
    protos_nb = F.normalize(protos_nb, dim=1)

    residuals_nb = torch.cat([
        e_support_nb[y_support_nb == c] - protos_nb[i]
        for i, c in enumerate(classes_nb)
    ], dim=0)

    D_nb = residuals_nb.shape[1]
    cov_nb = (residuals_nb.t() @ residuals_nb) / max(residuals_nb.shape[0] - 1, 1)

    avg_var_nb = torch.trace(cov_nb) / D_nb
    cov_shrink_nb = (1 - SHRINK) * cov_nb + SHRINK * (avg_var_nb * torch.eye(D_nb, device=device))
    cov_inv_nb = torch.linalg.pinv(cov_shrink_nb)

    diffs_nb = e_query_nb.unsqueeze(1) - protos_nb.unsqueeze(0)
    dists_nb = torch.einsum("nkd,dd,nkd->nk", diffs_nb, cov_inv_nb, diffs_nb)

    pred_nb_fs = dists_nb.argmin(dim=1)
    fewshot_acc_nb = (pred_nb_fs == y_query_nb).float().mean().item()

    print("N-back gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc_nb)

    # =====================================================
    # FEW-SHOT: HTC (GATED AdaBN + Mahalanobis)
    # =====================================================
    torch.manual_seed(2234 + fold_i)
    np.random.seed(2234 + fold_i)

    # start HTC adaptation from the same best joint model
    model.load_state_dict(best_joint_state)
    model.eval()

    all_e_htc = e_test_htc_base.clone()
    all_y_htc = y_test_htc_base.clone()

    support_idx_htc = []
    for c in torch.unique(all_y_htc):
        class_idx = (all_y_htc == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx_htc.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx_htc = torch.cat(support_idx_htc)

    mask_htc = torch.zeros(len(all_y_htc), dtype=torch.bool)
    mask_htc[support_idx_htc] = True

    support_subset_htc = torch.utils.data.Subset(test_ds_htc, support_idx_htc.tolist())
    support_loader_htc = DataLoader(support_subset_htc, batch_size=256, shuffle=False)

    use_bn_htc = shift_htc > SHIFT_THRESH
    print(f"HTC BN USED: {use_bn_htc}")

    if use_bn_htc:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader_htc:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract test embeddings after BN adaptation
        all_e_htc, all_y_htc = extract_embeddings(model, test_loader_htc, device)

    e_support_htc = all_e_htc[mask_htc].to(device)
    y_support_htc = all_y_htc[mask_htc].to(device)
    e_query_htc   = all_e_htc[~mask_htc].to(device)
    y_query_htc   = all_y_htc[~mask_htc].to(device)

    e_support_htc = F.normalize(e_support_htc, dim=1)
    e_query_htc   = F.normalize(e_query_htc, dim=1)

    var_htc = e_support_htc.var(dim=0)
    weights_htc = 1.0 / (var_htc + 1e-6)
    weights_htc = weights_htc / weights_htc.mean()
    e_support_htc = e_support_htc * weights_htc
    e_query_htc   = e_query_htc * weights_htc

    classes_htc = torch.unique(y_support_htc)
    protos_htc = torch.stack([
        e_support_htc[y_support_htc == c].mean(dim=0) for c in classes_htc
    ])
    protos_htc = F.normalize(protos_htc, dim=1)

    residuals_htc = torch.cat([
        e_support_htc[y_support_htc == c] - protos_htc[i]
        for i, c in enumerate(classes_htc)
    ], dim=0)

    D_htc = residuals_htc.shape[1]
    cov_htc = (residuals_htc.t() @ residuals_htc) / max(residuals_htc.shape[0] - 1, 1)

    avg_var_htc = torch.trace(cov_htc) / D_htc
    cov_shrink_htc = (1 - SHRINK) * cov_htc + SHRINK * (avg_var_htc * torch.eye(D_htc, device=device))
    cov_inv_htc = torch.linalg.pinv(cov_shrink_htc)

    diffs_htc = e_query_htc.unsqueeze(1) - protos_htc.unsqueeze(0)
    dists_htc = torch.einsum("nkd,dd,nkd->nk", diffs_htc, cov_inv_htc, diffs_htc)

    pred_htc_fs = dists_htc.argmin(dim=1)
    fewshot_acc_htc = (pred_htc_fs == y_query_htc).float().mean().item()

    print("HTC gated AdaBN + Mahalanobis few-shot acc:", fewshot_acc_htc)

    overall_results_joint.append({
        "fold": fold_i + 1,
        "test_subj_nb": test_subj_nb,
        "test_subj_htc": test_subj_htc,
        "best_zero_shot_nb": best_acc_nb,
        "best_zero_shot_htc": best_acc_htc,
        "shift_nb": shift_nb,
        "shift_htc": shift_htc,
        "bn_used_nb": bool(use_bn_nb),
        "bn_used_htc": bool(use_bn_htc),
        "fewshot_nb": fewshot_acc_nb,
        "fewshot_htc": fewshot_acc_htc
    })

print("\nJOINT METRIC TRAIN + GATED ADABN + MAHALANOBIS LOSO SUMMARY")
for r in overall_results_joint:
    print(
        f"Fold {r['fold']:02d} | "
        f"NB test {r['test_subj_nb']} | HTC test {r['test_subj_htc']} | "
        f"NB zero-shot {r['best_zero_shot_nb']:.4f} | "
        f"HTC zero-shot {r['best_zero_shot_htc']:.4f} | "
        f"NB shift {r['shift_nb']:.4f} | HTC shift {r['shift_htc']:.4f} | "
        f"NB BN {r['bn_used_nb']} | HTC BN {r['bn_used_htc']} | "
        f"NB few-shot {r['fewshot_nb']:.4f} | "
        f"HTC few-shot {r['fewshot_htc']:.4f}"
    )

nb_fs_accs  = [r["fewshot_nb"] for r in overall_results_joint]
htc_fs_accs = [r["fewshot_htc"] for r in overall_results_joint]

print("\nNB few-shot mean/std:", float(np.mean(nb_fs_accs)), float(np.std(nb_fs_accs)))
print("HTC few-shot mean/std:", float(np.mean(htc_fs_accs)), float(np.std(htc_fs_accs)))


JOINT FOLD 1/13
N-back test subject: subject_01
HTC test subject:    subject_01
Epoch 01 | NB zero-shot 0.6299 | HTC zero-shot 0.5896 | avg 0.6097
Epoch 02 | NB zero-shot 0.6614 | HTC zero-shot 0.6189 | avg 0.6402
Epoch 03 | NB zero-shot 0.6604 | HTC zero-shot 0.5212 | avg 0.5908
Epoch 04 | NB zero-shot 0.7140 | HTC zero-shot 0.5342 | avg 0.6241
Epoch 05 | NB zero-shot 0.7119 | HTC zero-shot 0.5277 | avg 0.6198
Epoch 06 | NB zero-shot 0.7245 | HTC zero-shot 0.5537 | avg 0.6391
Epoch 07 | NB zero-shot 0.7213 | HTC zero-shot 0.5407 | avg 0.6310
Epoch 08 | NB zero-shot 0.6824 | HTC zero-shot 0.5798 | avg 0.6311
Epoch 09 | NB zero-shot 0.6909 | HTC zero-shot 0.5635 | avg 0.6272
Epoch 10 | NB zero-shot 0.7697 | HTC zero-shot 0.5081 | avg 0.6389
Best-model shift | NB: 1.0610 | HTC: 2.5374
N-back BN USED: True
N-back gated AdaBN + Mahalanobis few-shot acc: 0.7654321193695068
HTC BN USED: True
HTC gated AdaBN + Mahalanobis few-shot acc: 0.7977527976036072

JOINT FOLD 2/13
N-back test subject:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np

def load_all_subjects_raw(subject_ids, base_path):
    # channel mapping
    model_channels = [
        'AF3','F7','F3','FC5',
        'T7','P7','O1','O2',
        'P8','T8','FC6','F4',
        'F8','AF4'
    ]

    source_names = [
        'FP1','AFF5','AFz','F1','FC5','FC1','T7','C3','Cz','CP5','CP1',
        'P7','P3','Pz','POz','O1',
        'FP2','AFF6','F2','FC2','FC6','C4','T8','CP2','CP6',
        'P4','P8','O2','HEOG','VEOG'
    ]

    mapping = {
        'AF3': 'AFF5',
        'F7' : 'FC5',
        'F3' : 'F1',
        'FC5': 'FC5',
        'T7' : 'T7',
        'P7' : 'P7',
        'O1' : 'O1',
        'O2' : 'O2',
        'P8' : 'P8',
        'T8' : 'T8',
        'FC6': 'FC6',
        'F4' : 'F2',
        'F8' : 'FC6',
        'AF4': 'AFF6'
    }

    indices = [source_names.index(mapping[ch]) for ch in model_channels]

    X_windows = []
    y_windows = []
    subjects = []

    for subj_id in subject_ids:
        data = np.load(f"{base_path}/subject{subj_id}_nback_eegnirsdataset.npz")

        X = data["X"][:, indices, :]   # (N, 14, 512)
        y = data["y"]

        X_windows.append(X)
        y_windows.append(y)
        subjects.append(np.array([f"subject_{subj_id:02d}"] * len(y)))

    X_windows = np.concatenate(X_windows, axis=0).astype(np.float32)
    y_windows = np.concatenate(y_windows)
    subjects  = np.concatenate(subjects)

    return X_windows, y_windows, subjects

In [ ]:
X_windows, y_windows, subjects = load_all_subjects_raw(
    subject_ids=range(1, 11),
    base_path="/content/drive/MyDrive"
)

In [ ]:
print("Starting bandpower computation")
print("X_windows dtype:", X_windows.dtype)
print("X_windows shape:", X_windows.shape)

FS = 128
bands = [
    ("theta", 4, 8),
    ("alpha", 8, 12),
    ("beta", 12, 30),
]

N, C, T = X_windows.shape
print("N:", N, "C:", C, "T:", T)

# ---- FFT frequencies ----
freqs = np.fft.rfftfreq(T, d=1.0/FS)
print("freqs shape:", freqs.shape)
print("First 10 freqs:", freqs[:10])
print("Last 10 freqs:", freqs[-10:])

# ---- FFT ----
X_fft = np.fft.rfft(X_windows, axis=-1)
print("X_fft shape:", X_fft.shape)
print("X_fft dtype:", X_fft.dtype)

# ---- PSD ----
PSD = (np.abs(X_fft) ** 2) / T
print("PSD shape:", PSD.shape)
print("PSD min/max:", PSD.min(), PSD.max())

# ---- Band extraction ----
bp_list = []

for name, f_lo, f_hi in bands:
    mask = (freqs >= f_lo) & (freqs < f_hi)

    print("\nBand:", name)
    print("Freq range:", f_lo, "-", f_hi)
    print("Number of bins in mask:", mask.sum())

    bp = PSD[..., mask]
    print("PSD slice shape:", bp.shape)

    bp_mean = bp.mean(axis=-1)
    print("After mean over freq axis:", bp_mean.shape)

    bp_list.append(bp_mean)

# ---- Stack bands ----
X_bp = np.stack(bp_list, axis=-1)
print("\nFinal stacked bandpower shape:", X_bp.shape)

# ---- Flatten ----
X_bp_flat = X_bp.reshape(N, -1).astype(np.float32)
print("Flattened shape:", X_bp_flat.shape)

print("Done bandpower.")

Starting bandpower computation
X_windows dtype: float32
X_windows shape: (2700, 14, 512)
N: 2700 C: 14 T: 512
freqs shape: (257,)
First 10 freqs: [0.   0.25 0.5  0.75 1.   1.25 1.5  1.75 2.   2.25]
Last 10 freqs: [61.75 62.   62.25 62.5  62.75 63.   63.25 63.5  63.75 64.  ]
X_fft shape: (2700, 14, 257)
X_fft dtype: complex64
PSD shape: (2700, 14, 257)
PSD min/max: 0.0 222.07425

Band: theta
Freq range: 4 - 8
Number of bins in mask: 16
PSD slice shape: (2700, 14, 16)
After mean over freq axis: (2700, 14)

Band: alpha
Freq range: 8 - 12
Number of bins in mask: 16
PSD slice shape: (2700, 14, 16)
After mean over freq axis: (2700, 14)

Band: beta
Freq range: 12 - 30
Number of bins in mask: 72
PSD slice shape: (2700, 14, 72)
After mean over freq axis: (2700, 14)

Final stacked bandpower shape: (2700, 14, 3)
Flattened shape: (2700, 42)
Done bandpower.


In [ ]:
print("Before relative normalization:")
print("Example band sums (first window, first channel):",
      X_bp[0, 0].sum())

# Relative within each channel
X_bp = X_bp / (X_bp.sum(axis=-1, keepdims=True) + 1e-8)

print("After relative normalization:")
print("Example band sums (first window, first channel):",
      X_bp[0, 0].sum())

# Flatten
N = X_bp.shape[0]
X_bp_flat = X_bp.reshape(N, -1).astype(np.float32)

print("X_bp_flat shape:", X_bp_flat.shape)

Before relative normalization:
Example band sums (first window, first channel): 1.2293341
After relative normalization:
Example band sums (first window, first channel): 1.0
X_bp_flat shape: (2700, 42)


In [ ]:
import torch

X_time_t = torch.tensor(X_windows, dtype=torch.float32)
X_bp_t   = torch.tensor(X_bp_flat, dtype=torch.float32)

unique_labels = np.unique(y_windows)
label_map = {v:i for i,v in enumerate(unique_labels)}
y_encoded = np.vectorize(label_map.get)(y_windows).astype(np.int64)

y_t = torch.tensor(y_encoded, dtype=torch.long)

print("X_time_t:", X_time_t.shape)
print("X_bp_t:", X_bp_t.shape)
print("y_t:", y_t.shape)
print("Label mapping:", label_map)

X_time_t: torch.Size([2700, 14, 512])
X_bp_t: torch.Size([2700, 42])
y_t: torch.Size([2700])
Label mapping: {np.int64(0): 0, np.int64(1): 1, np.int64(2): 2}


In [ ]:
N, C, T = X_windows.shape

cov_feats = []

for i in range(N):
    X = X_windows[i]
    X = X - X.mean(axis=1, keepdims=True)
    cov = (X @ X.T) / (T - 1)

    iu = np.triu_indices(C)
    cov_vec = cov[iu]
    cov_feats.append(cov_vec)

X_cov = np.array(cov_feats, dtype=np.float32)

print("X_cov shape:", X_cov.shape)

X_cov_t = torch.tensor(X_cov, dtype=torch.float32)

X_cov shape: (2700, 105)


In [ ]:
from scipy.signal import butter, filtfilt, hilbert

fs = 128
low, high = 4, 7

b, a = butter(4, [low/(fs/2), high/(fs/2)], btype='band')

N, C, T = X_windows.shape
plv_feats = []

for i in range(N):
    X = X_windows[i]

    X_filt = np.array([filtfilt(b, a, X[ch]) for ch in range(C)])

    analytic = hilbert(X_filt)
    phase = np.angle(analytic)

    plv_mat = np.zeros((C, C))
    for ch1 in range(C):
        for ch2 in range(ch1, C):
            phase_diff = phase[ch1] - phase[ch2]
            plv = np.abs(np.mean(np.exp(1j * phase_diff)))
            plv_mat[ch1, ch2] = plv
            plv_mat[ch2, ch1] = plv

    iu = np.triu_indices(C)
    plv_vec = plv_mat[iu]
    plv_feats.append(plv_vec)

X_plv = np.array(plv_feats, dtype=np.float32)
print("X_plv shape:", X_plv.shape)

X_plv_t = torch.tensor(X_plv, dtype=torch.float32)

X_plv shape: (2700, 105)


In [ ]:
low_a, high_a = 8, 12
b_a, a_a = butter(4, [low_a/(fs/2), high_a/(fs/2)], btype='band')

plv_alpha_feats = []

for i in range(N):
    X = X_windows[i]

    X_filt = np.array([filtfilt(b_a, a_a, X[ch]) for ch in range(C)])

    analytic = hilbert(X_filt)
    phase = np.angle(analytic)

    plv_mat = np.zeros((C, C))
    for ch1 in range(C):
        for ch2 in range(ch1, C):
            phase_diff = phase[ch1] - phase[ch2]
            plv = np.abs(np.mean(np.exp(1j * phase_diff)))
            plv_mat[ch1, ch2] = plv
            plv_mat[ch2, ch1] = plv

    iu = np.triu_indices(C)
    plv_alpha_feats.append(plv_mat[iu])

X_plv_alpha = np.array(plv_alpha_feats, dtype=np.float32)
print("Alpha PLV shape:", X_plv_alpha.shape)

X_plv_combined = np.concatenate([X_plv, X_plv_alpha], axis=1)
print("Combined PLV shape:", X_plv_combined.shape)

X_plv_t = torch.tensor(X_plv_combined, dtype=torch.float32)

Alpha PLV shape: (2700, 105)
Combined PLV shape: (2700, 210)


In [ ]:
class EEGDatasetNew(torch.utils.data.Dataset):
    def __init__(self, X_time, X_bp, X_cov, X_plv, y, subjects):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_cov  = X_cov
        self.X_plv  = X_plv
        self.y      = y

        # map subjects → integers
        uniq = np.unique(subjects)
        self.subj_map = {s:i for i,s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subjects])

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.X_bp[idx],
            self.X_cov[idx],
            self.X_plv[idx],
            torch.zeros(3),            # ROI placeholder
            self.y[idx],
            self.subj[idx],
            torch.tensor(1)            # dataset ID (same as NBACK)
        )

In [ ]:
class EEGDatasetNew(torch.utils.data.Dataset):
    def __init__(self, X_time, X_bp, X_cov, X_plv, y, subjects):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_cov  = X_cov
        self.X_plv  = X_plv
        self.y      = y

        # map subjects → integers
        uniq = np.unique(subjects)
        self.subj_map = {s:i for i,s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subjects])

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):

      return (
        self.X_time[idx],
        self.X_bp[idx],
        self.X_cov[idx],
        self.X_plv[idx],
        torch.zeros(0),        # ✅ MUST match training
        self.y[idx],
        self.subj[idx],
        torch.tensor(1)
    )

In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

EPOCHS = 10
BATCH  = 256
LR     = 1e-3
TEMP   = 0.1
PROTO_LAMBDA = 0.1

FEW_SHOT_PER_CLASS = 20
SHIFT_THRESH = 0.15

unique_subjects = np.unique(subjects)
results = []

# =========================
# SIMPLE PROTOTYPE LOSS
# =========================
def prototype_ce_loss(e, y, temperature=0.1):
    e = F.normalize(e, dim=1)
    classes = torch.unique(y)

    protos = torch.stack([
        e[y == c].mean(dim=0) for c in classes
    ])

    logits = e @ F.normalize(protos, dim=1).T

    label_map = {int(c.item()): i for i, c in enumerate(classes)}
    y_local = torch.tensor(
        [label_map[int(v.item())] for v in y],
        device=e.device
    )

    return F.cross_entropy(logits / temperature, y_local)

# =========================
# LOSO LOOP
# =========================
for fold_i, test_subj in enumerate(unique_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1} | TEST SUBJECT: {test_subj}")
    print("="*80)

    test_mask = (subjects == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    train_ds = EEGDatasetNew(
        X_time_t[train_idx],
        X_bp_t[train_idx],
        X_cov_t[train_idx],
        X_plv_t[train_idx],
        y_t[train_idx],
        subjects[train_idx]
    )

    test_ds = EEGDatasetNew(
        X_time_t[test_idx],
        X_bp_t[test_idx],
        X_cov_t[test_idx],
        X_plv_t[test_idx],
        y_t[test_idx],
        subjects[test_idx]
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_state = None
    best_acc = -1

    # =========================
    # TRAIN
    # =========================
    for ep in range(EPOCHS):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss = prototype_ce_loss(e, yb, TEMP)

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT VALIDATION
        # =========================
        model.eval()
        with torch.no_grad():
            e_train, y_train_full = extract_embeddings_fixed(model, train_loader, device)
            e_test,  y_test_full  = extract_embeddings_fixed(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([
                e_train[y_train_full == c].mean(dim=0) for c in classes
            ])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred.device
            )

            acc = (pred == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep+1} | val acc {acc:.4f}")

    # =========================
    # LOAD BEST MODEL
    # =========================
    model.load_state_dict(best_state)
    model.eval()

    # =========================
    # FEW-SHOT ON TEST SUBJECT
    # =========================
    e_test, y_test = extract_embeddings_fixed(model, test_loader, device)

    support_idx = []
    for c in torch.unique(y_test):
        idx = (y_test == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(idx))
        support_idx.append(idx[perm[:FEW_SHOT_PER_CLASS]])

    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(y_test), dtype=torch.bool)
    mask[support_idx] = True

    e_support = e_test[mask].to(device)
    y_support = y_test[mask].to(device)
    e_query   = e_test[~mask].to(device)
    y_query   = y_test[~mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    classes = torch.unique(y_support)
    protos = torch.stack([
        e_support[y_support == c].mean(dim=0) for c in classes
    ])

    logits = e_query @ F.normalize(protos, dim=1).T
    preds = logits.argmax(dim=1)

    acc = (preds == y_query).float().mean().item()
    print("🔥 LOSO Few-shot acc:", acc)

    results.append(acc)

# =========================
# FINAL
# =========================
print("\n==============================")
print("DATASET C LOSO RESULTS")
print("==============================")
print("Mean:", float(np.mean(results)))
print("Std :", float(np.std(results)))


FOLD 1 | TEST SUBJECT: subject_01
Epoch 1 | val acc 0.3630
Epoch 2 | val acc 0.3556
Epoch 3 | val acc 0.3667
Epoch 4 | val acc 0.3481
Epoch 5 | val acc 0.3370
Epoch 6 | val acc 0.3481
Epoch 7 | val acc 0.3593
Epoch 8 | val acc 0.3519
Epoch 9 | val acc 0.3630
Epoch 10 | val acc 0.3185
🔥 LOSO Few-shot acc: 0.5857142806053162

FOLD 2 | TEST SUBJECT: subject_02
Epoch 1 | val acc 0.5074
Epoch 2 | val acc 0.4296
Epoch 3 | val acc 0.4037
Epoch 4 | val acc 0.4037
Epoch 5 | val acc 0.4111
Epoch 6 | val acc 0.3407
Epoch 7 | val acc 0.3963
Epoch 8 | val acc 0.3889
Epoch 9 | val acc 0.3815
Epoch 10 | val acc 0.3741
🔥 LOSO Few-shot acc: 0.723809540271759

FOLD 3 | TEST SUBJECT: subject_03
Epoch 1 | val acc 0.5185
Epoch 2 | val acc 0.5074
Epoch 3 | val acc 0.4963
Epoch 4 | val acc 0.5111
Epoch 5 | val acc 0.4481
Epoch 6 | val acc 0.4259
Epoch 7 | val acc 0.4370
Epoch 8 | val acc 0.4370
Epoch 9 | val acc 0.4519
Epoch 10 | val acc 0.4741
🔥 LOSO Few-shot acc: 0.561904788017273

FOLD 4 | TEST SUBJECT: 

In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

EPOCHS = 10
BATCH  = 256
LR     = 1e-3

FEW_SHOT_PER_CLASS = 20
SHIFT_THRESH = 0.15
SHRINK = 0.4

unique_subjects = np.unique(subjects)
results = []

for fold_i, test_subj in enumerate(unique_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1} | TEST SUBJECT: {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask = (subjects == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    train_ds = EEGDatasetNew(
        X_time_t[train_idx],
        X_bp_t[train_idx],
        X_cov_t[train_idx],
        X_plv_t[train_idx],
        y_t[train_idx],
        subjects[train_idx]
    )

    test_ds = EEGDatasetNew(
        X_time_t[test_idx],
        X_bp_t[test_idx],
        X_cov_t[test_idx],
        X_plv_t[test_idx],
        y_t[test_idx],
        subjects[test_idx]
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
    train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_state = None
    best_acc = -1

    # =========================
    # TRAIN
    # =========================
    for ep in range(EPOCHS):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss = prototype_ce_loss(e, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

        # simple val
        model.eval()
        with torch.no_grad():
            e_train, y_train_full = extract_embeddings_fixed(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings_fixed(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([
                e_train[y_train_full == c].mean(dim=0) for c in classes
            ])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred.device
            )

            acc = (pred == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep+1} | val acc {acc:.4f}")

    # =========================
    # LOAD BEST
    # =========================
    model.load_state_dict(best_state)
    model.eval()

    # =========================
    # EMBEDDINGS
    # =========================
    e_train, _ = extract_embeddings_fixed(model, train_eval_loader, device)
    e_test, y_test = extract_embeddings_fixed(model, test_loader, device)

    # =========================
    # SHIFT
    # =========================
    shift = torch.norm(e_train.mean(0) - e_test.mean(0)).item()
    print("Shift:", shift)

    # =========================
    # FEW-SHOT SPLIT
    # =========================
    support_idx = []
    for c in torch.unique(y_test):
        idx = (y_test == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(idx))
        support_idx.append(idx[perm[:FEW_SHOT_PER_CLASS]])

    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(y_test), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    # =========================
    # BN ADAPTATION
    # =========================
    use_bn = shift > SHIFT_THRESH
    print("BN:", use_bn)

    if use_bn:
        for p in model.parameters():
            p.requires_grad = False

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, *_ in support_loader:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # recompute
        e_test, y_test = extract_embeddings_fixed(model, test_loader, device)

    # =========================
    # FEW-SHOT MAHALANOBIS
    # =========================
    e_support = e_test[mask].to(device)
    y_support = y_test[mask].to(device)
    e_query   = e_test[~mask].to(device)
    y_query   = y_test[~mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([
        e_support[y_support == c].mean(dim=0)
        for c in classes
    ])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([
        e_support[y_support == c] - protos[i]
        for i, c in enumerate(classes)
    ], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0]-1, 1)

    avg_var = torch.trace(cov) / D
    cov = (1-SHRINK)*cov + SHRINK*(avg_var*torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    preds = dists.argmin(dim=1)
    acc = (preds == y_query).float().mean().item()

    print("🔥 LOSO Few-shot acc:", acc)
    results.append(acc)

# =========================
# FINAL
# =========================
print("\n==============================")
print("DATASET C TRUE LOSO RESULTS")
print("==============================")
print("Mean:", float(np.mean(results)))
print("Std :", float(np.std(results)))


FOLD 1 | TEST SUBJECT: subject_01
Epoch 1 | val acc 0.3889
Epoch 2 | val acc 0.3407
Epoch 3 | val acc 0.4000
Epoch 4 | val acc 0.3926
Epoch 5 | val acc 0.3963
Epoch 6 | val acc 0.4074
Epoch 7 | val acc 0.4000
Epoch 8 | val acc 0.4185
Epoch 9 | val acc 0.4037
Epoch 10 | val acc 0.3815
Shift: 0.9966880679130554
BN: True
🔥 LOSO Few-shot acc: 0.5190476179122925

FOLD 2 | TEST SUBJECT: subject_02
Epoch 1 | val acc 0.3926
Epoch 2 | val acc 0.3889
Epoch 3 | val acc 0.4259
Epoch 4 | val acc 0.3778
Epoch 5 | val acc 0.4296
Epoch 6 | val acc 0.4370
Epoch 7 | val acc 0.4185
Epoch 8 | val acc 0.3778
Epoch 9 | val acc 0.3889
Epoch 10 | val acc 0.5370
Shift: 1.4036825895309448
BN: True
🔥 LOSO Few-shot acc: 0.6714286208152771

FOLD 3 | TEST SUBJECT: subject_03
Epoch 1 | val acc 0.4926
Epoch 2 | val acc 0.5037
Epoch 3 | val acc 0.4889
Epoch 4 | val acc 0.4815
Epoch 5 | val acc 0.4741
Epoch 6 | val acc 0.4444
Epoch 7 | val acc 0.4704
Epoch 8 | val acc 0.4481
Epoch 9 | val acc 0.4889
Epoch 10 | val acc

In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4
SHIFT_THRESH = 0.15

all_subjects_C = np.unique(subjects)
results = []

# =========================
# LOSS (same style, single dataset)
# =========================
def metric_loss(e, proj, y):
    e = F.normalize(e, dim=1)
    proj = F.normalize(proj, dim=1)

    classes = torch.unique(y)

    protos = torch.stack([
        e[y == c].mean(dim=0) for c in classes
    ])

    logits = e @ F.normalize(protos, dim=1).T

    label_map = {int(c.item()): i for i, c in enumerate(classes)}
    y_local = torch.tensor(
        [label_map[int(v.item())] for v in y],
        device=e.device
    )

    loss_proto = F.cross_entropy(logits / TEMP, y_local)

    # contrastive (same spirit)
    sim = proj @ proj.T
    mask = y.unsqueeze(1) == y.unsqueeze(0)
    loss_con = -torch.log(
        (torch.exp(sim / TEMP) * mask).sum(1) /
        torch.exp(sim / TEMP).sum(1)
    ).mean()

    return loss_proto + PROTO_LAMBDA * loss_con


# =========================
# LOSO LOOP
# =========================
for fold_i, test_subj in enumerate(all_subjects_C):

    print("\n" + "="*90)
    print(f"FOLD {fold_i+1} | TEST SUBJECT: {test_subj}")
    print("="*90)

    test_mask = (subjects == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # SPLIT DATA
    # =========================
    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    X_cov_train = X_cov_t[train_idx].clone()
    X_cov_test  = X_cov_t[test_idx].clone()

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    # =========================
    # NORMALIZATION (SAME AS YOU)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    X_roi_train = torch.zeros((len(train_idx), 0))
    X_roi_test  = torch.zeros((len(test_idx), 0))

    # =========================
    # DATASETS
    # =========================
    train_ds = EEGDataset(
        X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train,
        y_train,
        subjects[train_idx],
        dataset_id=0
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test, X_cov_test, X_plv_test, X_roi_test,
        y_test,
        subjects[test_idx],
        dataset_id=0
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
    train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_state = None
    best_acc = -1

    # =========================
    # TRAIN
    # =========================
    for ep in range(EPOCHS):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss = metric_loss(e, proj, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # VALIDATION (same style)
        # =========================
        model.eval()
        with torch.no_grad():
            e_train, y_train_full = extract_embeddings_fixed(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings_fixed(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([
                e_train[y_train_full == c].mean(dim=0) for c in classes
            ])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred.device
            )

            acc = (pred == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep+1} | val acc {acc:.4f}")

    # =========================
    # LOAD BEST
    # =========================
    model.load_state_dict(best_state)
    model.eval()

    # =========================
    # EMBEDDINGS
    # =========================
    e_train, _ = extract_embeddings_fixed(model, train_eval_loader, device)
    e_test, y_test = extract_embeddings_fixed(model, test_loader, device)

    shift = torch.norm(e_train.mean(0) - e_test.mean(0)).item()
    print("Shift:", shift)

    # =========================
    # FEW-SHOT SPLIT
    # =========================
    support_idx = []
    for c in torch.unique(y_test):
        idx = (y_test == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(idx))
        support_idx.append(idx[:FEW_SHOT_PER_CLASS])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(y_test), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    # =========================
    # BN ADAPTATION
    # =========================
    use_bn = shift > SHIFT_THRESH
    print("BN:", use_bn)

    if use_bn:
        for p in model.parameters():
            p.requires_grad = False

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, *_ in support_loader:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        e_test, y_test = extract_embeddings_fixed(model, test_loader, device)

    # =========================
    # MAHALANOBIS FEW-SHOT
    # =========================
    e_support = e_test[mask].to(device)
    y_support = y_test[mask].to(device)
    e_query   = e_test[~mask].to(device)
    y_query   = y_test[~mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support *= weights
    e_query   *= weights

    classes = torch.unique(y_support)
    protos = torch.stack([
        e_support[y_support == c].mean(dim=0)
        for c in classes
    ])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([
        e_support[y_support == c] - protos[i]
        for i, c in enumerate(classes)
    ], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0]-1,1)

    avg_var = torch.trace(cov)/D
    cov = (1-SHRINK)*cov + SHRINK*(avg_var*torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    preds = dists.argmin(dim=1)
    acc = (preds == y_query).float().mean().item()

    print("🔥 LOSO Few-shot acc:", acc)
    results.append(acc)

# =========================
# FINAL
# =========================
print("\n==============================")
print("DATASET C TRUE LOSO RESULTS")
print("==============================")
print("Mean:", float(np.mean(results)))
print("Std :", float(np.std(results)))


FOLD 1 | TEST SUBJECT: subject_01
Epoch 1 | val acc 0.4148
Epoch 2 | val acc 0.4148
Epoch 3 | val acc 0.4074
Epoch 4 | val acc 0.3963
Epoch 5 | val acc 0.4037
Epoch 6 | val acc 0.4074
Epoch 7 | val acc 0.4037
Epoch 8 | val acc 0.4111
Epoch 9 | val acc 0.4185
Epoch 10 | val acc 0.4000
Shift: 1.206164836883545
BN: True
🔥 LOSO Few-shot acc: 0.5190476179122925

FOLD 2 | TEST SUBJECT: subject_02
Epoch 1 | val acc 0.3926
Epoch 2 | val acc 0.4852
Epoch 3 | val acc 0.4444
Epoch 4 | val acc 0.4407
Epoch 5 | val acc 0.4259
Epoch 6 | val acc 0.4556
Epoch 7 | val acc 0.4593
Epoch 8 | val acc 0.4593
Epoch 9 | val acc 0.4556
Epoch 10 | val acc 0.4667
Shift: 0.3217620253562927
BN: True
🔥 LOSO Few-shot acc: 0.6761904954910278

FOLD 3 | TEST SUBJECT: subject_03
Epoch 1 | val acc 0.5259
Epoch 2 | val acc 0.5037
Epoch 3 | val acc 0.4963
Epoch 4 | val acc 0.5074
Epoch 5 | val acc 0.4963
Epoch 6 | val acc 0.5000
Epoch 7 | val acc 0.4778
Epoch 8 | val acc 0.4963
Epoch 9 | val acc 0.4704
Epoch 10 | val acc 

In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

overall_results = []
all_subjects_C = np.unique(subjects)

for fold_i, test_subj in enumerate(all_subjects_C):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects_C)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    # keep same behavior as your old single-dataset loop:
    # covariance branch disabled
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0), dtype=torch.float32)
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0), dtype=torch.float32)

    # =========================
    # DATASET / LOADER
    # =========================
    # This matches your current dataset class shape:
    # (X_time, X_bp, X_cov, X_plv, y, subjects)
    train_ds = EEGDatasetNew(
        X_time_train, X_bp_train, X_cov_train, X_plv_train,
        y_train, subj_train
    )
    test_ds = EEGDatasetNew(
        X_time_test, X_bp_test, X_cov_test, X_plv_test,
        y_test, subj_test
    )

    # Use the same sampler idea if it exists in your notebook.
    sampler = SubjectBalancedBatchSampler(
        train_ds,
        batch_size=BATCH,
        subjects_per_batch=12
    )
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=TEMP)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([
                e_train[y_train_full == c].mean(dim=0) for c in classes
            ])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # =========================
    # LOAD BEST ENCODER
    # =========================
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # =====================================
    # FEATURE RELIABILITY WEIGHTING
    # =====================================
    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()
    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([
        e_support[y_support == c].mean(dim=0) for c in classes
    ])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([
        e_support[y_support == c] - protos[i]
        for i, c in enumerate(classes)
    ], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))

print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/10 | TEST SUBJECT = subject_01


ValueError: Sample larger than population or is negative

In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4
SHIFT_THRESH = 0.15

unique_subjects = np.unique(subjects)
results = []

# dataset_C already created from your tensors
dataset_C = EEGDatasetNew(
    X_time_t, X_bp_t, X_cov_t, X_plv_t,
    y_t,
    subjects
)

def extract_embeddings_fixed(model, loader, device):
    model.eval()
    E, Y = [], []
    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)
            E.append(e.detach().cpu())
            Y.append(yb.detach().cpu())

    return torch.cat(E, dim=0), torch.cat(Y, dim=0)

for subj_i, subj in enumerate(unique_subjects):
    print("\n" + "=" * 80)
    print(f"SUBJECT TEST: {subj}")
    print("=" * 80)

    # restore trained model each subject
    model.load_state_dict(best_joint_state)   # or your saved trained weights
    model.eval()

    subj_idx = np.where(subjects == subj)[0]
    subj_subset = Subset(dataset_C, subj_idx.tolist())
    subj_loader = DataLoader(subj_subset, batch_size=256, shuffle=False)

    # embeddings before adaptation
    e_subj, y_subj = extract_embeddings_fixed(model, subj_loader, device)

    # few-shot split WITHIN subject
    torch.manual_seed(1234 + subj_i)
    np.random.seed(1234 + subj_i)

    support_idx_local = []
    for c in torch.unique(y_subj):
        class_idx = (y_subj == c).nonzero(as_tuple=True)[0]

        if len(class_idx) < FEW_SHOT_PER_CLASS:
            print(f"Skipping class {int(c)} for {subj}: only {len(class_idx)} samples")
            continue

        perm = torch.randperm(len(class_idx))
        support_idx_local.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])

    # skip subject if any class is missing too much data
    n_classes_present = len(torch.unique(y_subj))
    if len(support_idx_local) != n_classes_present:
        print(f"Skipping subject {subj}: not enough samples per class")
        continue

    support_idx_local = torch.cat(support_idx_local)

    mask = torch.zeros(len(y_subj), dtype=torch.bool)
    mask[support_idx_local] = True

    # optional shift estimate
    shift = torch.norm(e_subj.mean(0) - e_subj[support_idx_local].mean(0)).item()
    print("Shift:", shift)

    # BN adaptation using THIS SUBJECT'S support only
    use_bn = shift > SHIFT_THRESH
    print("BN ADAPTATION:", "ON" if use_bn else "OFF")

    if use_bn:
        for p in model.parameters():
            p.requires_grad = False

        model.eval()
        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        support_subset = Subset(subj_subset, support_idx_local.tolist())
        support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in support_loader:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # re-extract after BN adaptation
        e_subj, y_subj = extract_embeddings_fixed(model, subj_loader, device)

    e_support = e_subj[mask].to(device)
    y_support = y_subj[mask].to(device)
    e_query   = e_subj[~mask].to(device)
    y_query   = y_subj[~mask].to(device)

    if len(y_query) == 0:
        print(f"Skipping subject {subj}: no query samples left")
        continue

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([
        e_support[y_support == c].mean(dim=0)
        for c in classes
    ])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([
        e_support[y_support == c] - protos[i]
        for i, c in enumerate(classes)
    ], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    acc = (pred == y_query).float().mean().item()

    print("Few-shot acc:", acc)

    results.append({
        "subject": subj,
        "shift": shift,
        "bn_used": bool(use_bn),
        "acc": acc,
        "n_support": int(mask.sum().item()),
        "n_query": int((~mask).sum().item()),
    })

print("\n" + "=" * 80)
print("DATASET C SUBJECT-WISE FEW-SHOT RESULTS")
print("=" * 80)
for r in results:
    print(
        f"{r['subject']} | "
        f"shift {r['shift']:.4f} | "
        f"BN {r['bn_used']} | "
        f"acc {r['acc']:.4f} | "
        f"support {r['n_support']} | "
        f"query {r['n_query']}"
    )

if len(results) > 0:
    accs = [r["acc"] for r in results]
    print("\nMean:", float(np.mean(accs)))
    print("Std :", float(np.std(accs)))


SUBJECT TEST: subject_01
Shift: 0.4355091452598572
BN ADAPTATION: ON
Few-shot acc: 0.5047619342803955

SUBJECT TEST: subject_02
Shift: 0.16991297900676727
BN ADAPTATION: ON
Few-shot acc: 0.5714285969734192

SUBJECT TEST: subject_03
Shift: 0.21728719770908356
BN ADAPTATION: ON
Few-shot acc: 0.4571428894996643

SUBJECT TEST: subject_04
Shift: 0.320478618144989
BN ADAPTATION: ON
Few-shot acc: 0.43809524178504944

SUBJECT TEST: subject_05
Shift: 0.17439694702625275
BN ADAPTATION: ON
Few-shot acc: 0.3857142925262451

SUBJECT TEST: subject_06
Shift: 0.2586807906627655
BN ADAPTATION: ON
Few-shot acc: 0.37142857909202576

SUBJECT TEST: subject_07
Shift: 0.31286463141441345
BN ADAPTATION: ON
Few-shot acc: 0.380952388048172

SUBJECT TEST: subject_08
Shift: 0.17150261998176575
BN ADAPTATION: ON
Few-shot acc: 0.49523812532424927

SUBJECT TEST: subject_09
Shift: 0.18456648290157318
BN ADAPTATION: ON
Few-shot acc: 0.4047619104385376

SUBJECT TEST: subject_10
Shift: 0.1712539941072464
BN ADAPTATION:

In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

FEW_SHOT_PER_CLASS = 20
SHIFT_THRESH = 0.15

# =========================
# DATASET
# =========================
dataset_C = EEGDatasetNew(
    X_time_t, X_bp_t, X_cov_t, X_plv_t,
    y_t,
    subjects
)

# =========================
# EMBEDDING FUNCTION
# =========================
def extract_embeddings_fixed(model, loader, device):
    model.eval()
    E, Y = [], []
    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)
            E.append(e.detach().cpu())
            Y.append(yb.detach().cpu())

    return torch.cat(E, dim=0), torch.cat(Y, dim=0)

# =========================
# REFERENCE EMBEDDINGS (IMPORTANT FIX)
# =========================
# use your training data loader here (NB or HTC)
e_ref, _ = extract_embeddings_fixed(model, train_eval_loader_nb, device)
ref_mean = e_ref.mean(0)

# =========================
# SUBJECT-WISE EVAL
# =========================
unique_subjects = np.unique(subjects)
results = []

for subj_i, subj in enumerate(unique_subjects):

    print("\n" + "=" * 80)
    print(f"SUBJECT TEST: {subj}")
    print("=" * 80)

    # restore trained model
    model.load_state_dict(best_joint_state)
    model.eval()

    subj_idx = np.where(subjects == subj)[0]
    subj_subset = Subset(dataset_C, subj_idx.tolist())
    subj_loader = DataLoader(subj_subset, batch_size=256, shuffle=False)

    # embeddings
    e_subj, y_subj = extract_embeddings_fixed(model, subj_loader, device)

    # =========================
    # SHIFT (FIXED)
    # =========================
    shift = torch.norm(ref_mean - e_subj.mean(0)).item()
    print("Shift:", shift)

    # =========================
    # FEW-SHOT SPLIT
    # =========================
    torch.manual_seed(1234 + subj_i)
    np.random.seed(1234 + subj_i)

    support_idx_local = []
    for c in torch.unique(y_subj):
        class_idx = (y_subj == c).nonzero(as_tuple=True)[0]

        if len(class_idx) < FEW_SHOT_PER_CLASS:
            print(f"Skipping class {int(c)} for {subj}")
            continue

        perm = torch.randperm(len(class_idx))
        support_idx_local.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])

    if len(support_idx_local) != len(torch.unique(y_subj)):
        print(f"Skipping subject {subj}")
        continue

    support_idx_local = torch.cat(support_idx_local)

    mask = torch.zeros(len(y_subj), dtype=torch.bool)
    mask[support_idx_local] = True

    # =========================
    # BN ADAPTATION
    # =========================
    use_bn = shift > SHIFT_THRESH
    print("BN ADAPTATION:", "ON" if use_bn else "OFF")

    if use_bn:
        for p in model.parameters():
            p.requires_grad = False

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        support_subset = Subset(subj_subset, support_idx_local.tolist())
        support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, *_ in support_loader:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # recompute embeddings
        e_subj, y_subj = extract_embeddings_fixed(model, subj_loader, device)

    # =========================
    # FEW-SHOT CLASSIFICATION (SIMPLIFIED)
    # =========================
    e_support = e_subj[mask].to(device)
    y_support = y_subj[mask].to(device)

    e_query   = e_subj[~mask].to(device)
    y_query   = y_subj[~mask].to(device)

    if len(y_query) == 0:
        print("No query samples")
        continue

    # normalize
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # prototypes
    classes = torch.unique(y_support)
    protos = torch.stack([
        e_support[y_support == c].mean(dim=0)
        for c in classes
    ])

    # Euclidean distance
    dists = torch.cdist(e_query, protos)
    preds = dists.argmin(dim=1)

    acc = (preds == y_query).float().mean().item()
    print("🔥 Few-shot acc:", acc)

    results.append(acc)

# =========================
# FINAL RESULTS
# =========================
print("\n==============================")
print("FINAL RESULTS")
print("==============================")

if len(results) > 0:
    print("Mean:", float(np.mean(results)))
    print("Std :", float(np.std(results)))


SUBJECT TEST: subject_01
Shift: 11.045785903930664
BN ADAPTATION: ON
🔥 Few-shot acc: 0.485714316368103

SUBJECT TEST: subject_02
Shift: 10.52341365814209
BN ADAPTATION: ON
🔥 Few-shot acc: 0.5523809790611267

SUBJECT TEST: subject_03
Shift: 9.814065933227539
BN ADAPTATION: ON
🔥 Few-shot acc: 0.43809524178504944

SUBJECT TEST: subject_04
Shift: 13.164510726928711
BN ADAPTATION: ON
🔥 Few-shot acc: 0.44285714626312256

SUBJECT TEST: subject_05
Shift: 12.419561386108398
BN ADAPTATION: ON
🔥 Few-shot acc: 0.3857142925262451

SUBJECT TEST: subject_06
Shift: 10.510234832763672
BN ADAPTATION: ON
🔥 Few-shot acc: 0.39047619700431824

SUBJECT TEST: subject_07
Shift: 9.209190368652344
BN ADAPTATION: ON
🔥 Few-shot acc: 0.3857142925262451

SUBJECT TEST: subject_08
Shift: 9.664539337158203
BN ADAPTATION: ON
🔥 Few-shot acc: 0.46666669845581055

SUBJECT TEST: subject_09
Shift: 9.918540954589844
BN ADAPTATION: ON
🔥 Few-shot acc: 0.4047619104385376

SUBJECT TEST: subject_10
Shift: 9.968714714050293
BN ADA

In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

FEW_SHOT_PER_CLASS = 20
SHIFT_THRESH = 0.15
FT_STEPS = 8
FT_LR = 1e-4

# =========================
# DATASET
# =========================
dataset_C = EEGDatasetNew(
    X_time_t, X_bp_t, X_cov_t, X_plv_t,
    y_t,
    subjects
)

# =========================
# EMBEDDING EXTRACTOR
# =========================
def extract_embeddings_fixed(model, loader, device):
    model.eval()
    E, Y = [], []
    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)
            E.append(e.detach().cpu())
            Y.append(yb.detach().cpu())

    return torch.cat(E, dim=0), torch.cat(Y, dim=0)

# =========================
# REFERENCE EMBEDDINGS
# =========================
# This assumes train_eval_loader_nb exists in your notebook.
# If you want, you can later replace this with a combined NB+HTC reference.
e_ref, _ = extract_embeddings_fixed(model, train_eval_loader_nb, device)
ref_mean = e_ref.mean(0)

# =========================
# SUBJECT LOOP
# =========================
unique_subjects = np.unique(subjects)
results = []

for subj_i, subj in enumerate(unique_subjects):

    print("\n" + "=" * 80)
    print(f"SUBJECT TEST: {subj}")
    print("=" * 80)

    # restore trained model
    model.load_state_dict(best_joint_state)
    model.eval()

    subj_idx = np.where(subjects == subj)[0]
    subj_subset = Subset(dataset_C, subj_idx.tolist())
    subj_loader = DataLoader(subj_subset, batch_size=256, shuffle=False)

    # initial embeddings
    e_subj, y_subj = extract_embeddings_fixed(model, subj_loader, device)

    # =========================
    # SHIFT
    # =========================
    shift = torch.norm(ref_mean - e_subj.mean(0)).item()
    print("Shift:", shift)

    # =========================
    # FEW-SHOT SPLIT
    # =========================
    torch.manual_seed(1234 + subj_i)
    np.random.seed(1234 + subj_i)

    support_idx_local = []
    unique_classes = torch.unique(y_subj)

    for c in unique_classes:
        class_idx = (y_subj == c).nonzero(as_tuple=True)[0]

        if len(class_idx) < FEW_SHOT_PER_CLASS:
            print(f"Skipping class {int(c)} for {subj}: only {len(class_idx)} samples")
            continue

        perm = torch.randperm(len(class_idx))
        support_idx_local.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])

    if len(support_idx_local) != len(unique_classes):
        print(f"Skipping subject {subj}: insufficient samples per class")
        continue

    support_idx_local = torch.cat(support_idx_local)

    mask = torch.zeros(len(y_subj), dtype=torch.bool)
    mask[support_idx_local] = True

    # =========================
    # BN ADAPTATION
    # =========================
    use_bn = shift > SHIFT_THRESH
    print("BN ADAPTATION:", "ON" if use_bn else "OFF")

    if use_bn:
        for p in model.parameters():
            p.requires_grad = False

        set_partial_bn_adapt(
            model,
            allow=("encoder.time_branch",
                   "encoder.bp_branch",
                   "encoder.cov_branch",
                   "encoder.plv_branch")
        )

        support_subset = Subset(subj_subset, support_idx_local.tolist())
        support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, *_ in support_loader:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # recompute embeddings after BN adaptation
        e_subj, y_subj = extract_embeddings_fixed(model, subj_loader, device)

    # =========================
    # FEW-SHOT FINETUNE
    # =========================
    model.train()

    # Re-enable grads everywhere first
    for p in model.parameters():
        p.requires_grad = True

    # Optional: freeze only BN params if you want to preserve adapted stats.
    # Safe default here: fine-tune everything that contributes to e.
    optimizer_ft = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=FT_LR
    )

    support_subset = Subset(subj_subset, support_idx_local.tolist())
    support_loader = DataLoader(support_subset, batch_size=64, shuffle=True)

    for _ in range(FT_STEPS):
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, *_ in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)

            e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            # IMPORTANT: no normalization here
            classes = torch.unique(yb)
            protos = torch.stack([
                e[yb == c].mean(dim=0) for c in classes
            ])

            logits = e @ F.normalize(protos, dim=1).T

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_local = torch.tensor(
                [label_map[int(v.item())] for v in yb],
                device=device
            )

            loss = F.cross_entropy(logits, y_local)

            optimizer_ft.zero_grad()
            loss.backward()
            optimizer_ft.step()

    # =========================
    # FINAL EMBEDDINGS
    # =========================
    model.eval()
    e_subj, y_subj = extract_embeddings_fixed(model, subj_loader, device)

    e_support = e_subj[mask].to(device)
    y_support = y_subj[mask].to(device)
    e_query   = e_subj[~mask].to(device)
    y_query   = y_subj[~mask].to(device)

    if len(y_query) == 0:
        print("No query samples")
        continue

    # =========================
    # FINAL CLASSIFIER
    # =========================
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    classes = torch.unique(y_support)
    protos = torch.stack([
        e_support[y_support == c].mean(dim=0)
        for c in classes
    ])

    logits = e_query @ F.normalize(protos, dim=1).T
    preds = logits.argmax(dim=1)

    acc = (preds == y_query).float().mean().item()
    print("🔥 Few-shot + FT acc:", acc)

    results.append({
        "subject": subj,
        "shift": shift,
        "bn_used": bool(use_bn),
        "acc": acc,
        "n_support": int(mask.sum().item()),
        "n_query": int((~mask).sum().item()),
    })

# =========================
# FINAL RESULTS
# =========================
print("\n" + "=" * 80)
print("FINAL RESULTS")
print("=" * 80)

for r in results:
    print(
        f"{r['subject']} | "
        f"shift {r['shift']:.4f} | "
        f"BN {r['bn_used']} | "
        f"acc {r['acc']:.4f} | "
        f"support {r['n_support']} | "
        f"query {r['n_query']}"
    )

if len(results) > 0:
    accs = [r["acc"] for r in results]
    print("\nMean:", float(np.mean(accs)))
    print("Std :", float(np.std(accs)))


SUBJECT TEST: subject_01
Shift: 10.99954891204834
BN ADAPTATION: ON
🔥 Few-shot + FT acc: 0.5047619342803955

SUBJECT TEST: subject_02
Shift: 10.480415344238281
BN ADAPTATION: ON
🔥 Few-shot + FT acc: 0.5809524059295654

SUBJECT TEST: subject_03
Shift: 9.798699378967285
BN ADAPTATION: ON
🔥 Few-shot + FT acc: 0.4095238149166107

SUBJECT TEST: subject_04
Shift: 13.07896900177002
BN ADAPTATION: ON
🔥 Few-shot + FT acc: 0.4476190507411957

SUBJECT TEST: subject_05
Shift: 12.348590850830078
BN ADAPTATION: ON
🔥 Few-shot + FT acc: 0.4047619104385376

SUBJECT TEST: subject_06
Shift: 10.457063674926758
BN ADAPTATION: ON
🔥 Few-shot + FT acc: 0.4047619104385376

SUBJECT TEST: subject_07
Shift: 9.250327110290527
BN ADAPTATION: ON
🔥 Few-shot + FT acc: 0.39047619700431824

SUBJECT TEST: subject_08
Shift: 9.602807998657227
BN ADAPTATION: ON
🔥 Few-shot + FT acc: 0.43809524178504944

SUBJECT TEST: subject_09
Shift: 9.842305183410645
BN ADAPTATION: ON
🔥 Few-shot + FT acc: 0.39047619700431824

SUBJECT TEST

In [ ]:
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

FEW_SHOT_PER_CLASS = 20
FT_STEPS = 8
FT_LR = 1e-4

# =========================
# DATASET
# =========================
dataset_C = EEGDatasetNew(
    X_time_t, X_bp_t, X_cov_t, X_plv_t,
    y_t,
    subjects
)

# =========================
# EMBEDDING EXTRACTOR
# =========================
def extract_embeddings_fixed(model, loader, device):
    model.eval()
    E, Y = [], []
    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)
            E.append(e.detach().cpu())
            Y.append(yb.detach().cpu())

    return torch.cat(E, dim=0), torch.cat(Y, dim=0)

# =========================
# REFERENCE EMBEDDINGS
# =========================
e_ref, _ = extract_embeddings_fixed(model, train_eval_loader_nb, device)
ref_mean = e_ref.mean(0)

# =========================
# SUBJECT LOOP
# =========================
unique_subjects = np.unique(subjects)
results = []

for subj_i, subj in enumerate(unique_subjects):

    print("\n" + "=" * 80)
    print(f"SUBJECT TEST: {subj}")
    print("=" * 80)

    # restore trained model
    model.load_state_dict(best_joint_state)
    model.eval()

    subj_idx = np.where(subjects == subj)[0]
    subj_subset = Subset(dataset_C, subj_idx.tolist())
    subj_loader = DataLoader(subj_subset, batch_size=256, shuffle=False)

    # initial embeddings
    e_subj, y_subj = extract_embeddings_fixed(model, subj_loader, device)

    # =========================
    # SHIFT
    # =========================
    shift = torch.norm(ref_mean - e_subj.mean(0)).item()
    print("Shift:", shift)

    # =========================
    # FEW-SHOT SPLIT
    # =========================
    torch.manual_seed(1234 + subj_i)
    np.random.seed(1234 + subj_i)

    support_idx_local = []
    unique_classes = torch.unique(y_subj)

    for c in unique_classes:
        class_idx = (y_subj == c).nonzero(as_tuple=True)[0]

        if len(class_idx) < FEW_SHOT_PER_CLASS:
            print(f"Skipping class {int(c)} for {subj}: only {len(class_idx)} samples")
            continue

        perm = torch.randperm(len(class_idx))
        support_idx_local.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])

    if len(support_idx_local) != len(unique_classes):
        print(f"Skipping subject {subj}: insufficient samples per class")
        continue

    support_idx_local = torch.cat(support_idx_local)

    mask = torch.zeros(len(y_subj), dtype=torch.bool)
    mask[support_idx_local] = True

    # =========================
    # BN ADAPTATION OFF
    # =========================
    use_bn = False
    print("BN ADAPTATION:", "ON" if use_bn else "OFF")

    # =========================
    # FEW-SHOT FINETUNE
    # =========================
    model.train()

    for p in model.parameters():
        p.requires_grad = True

    optimizer_ft = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=FT_LR
    )

    support_subset = Subset(subj_subset, support_idx_local.tolist())
    support_loader = DataLoader(support_subset, batch_size=64, shuffle=True)

    for _ in range(FT_STEPS):
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, *_ in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)

            e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            classes = torch.unique(yb)
            protos = torch.stack([
                e[yb == c].mean(dim=0) for c in classes
            ])

            logits = e @ F.normalize(protos, dim=1).T

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_local = torch.tensor(
                [label_map[int(v.item())] for v in yb],
                device=device
            )

            loss = F.cross_entropy(logits, y_local)

            optimizer_ft.zero_grad()
            loss.backward()
            optimizer_ft.step()

    # =========================
    # FINAL EMBEDDINGS
    # =========================
    model.eval()
    e_subj, y_subj = extract_embeddings_fixed(model, subj_loader, device)

    e_support = e_subj[mask].to(device)
    y_support = y_subj[mask].to(device)
    e_query   = e_subj[~mask].to(device)
    y_query   = y_subj[~mask].to(device)

    if len(y_query) == 0:
        print("No query samples")
        continue

    # =========================
    # FINAL CLASSIFIER
    # =========================
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    classes = torch.unique(y_support)
    protos = torch.stack([
        e_support[y_support == c].mean(dim=0)
        for c in classes
    ])

    logits = e_query @ F.normalize(protos, dim=1).T
    preds = logits.argmax(dim=1)

    acc = (preds == y_query).float().mean().item()
    print("🔥 Few-shot + FT acc:", acc)

    results.append({
        "subject": subj,
        "shift": shift,
        "bn_used": bool(use_bn),
        "acc": acc,
        "n_support": int(mask.sum().item()),
        "n_query": int((~mask).sum().item()),
    })

# =========================
# FINAL RESULTS
# =========================
print("\n" + "=" * 80)
print("FINAL RESULTS")
print("=" * 80)

for r in results:
    print(
        f"{r['subject']} | "
        f"shift {r['shift']:.4f} | "
        f"BN {r['bn_used']} | "
        f"acc {r['acc']:.4f} | "
        f"support {r['n_support']} | "
        f"query {r['n_query']}"
    )

if len(results) > 0:
    accs = [r["acc"] for r in results]
    print("\nMean:", float(np.mean(accs)))
    print("Std :", float(np.std(accs)))


SUBJECT TEST: subject_01
Shift: 11.086467742919922
BN ADAPTATION: OFF
🔥 Few-shot + FT acc: 0.5047619342803955

SUBJECT TEST: subject_02
Shift: 10.61341667175293
BN ADAPTATION: OFF
🔥 Few-shot + FT acc: 0.6047618985176086

SUBJECT TEST: subject_03
Shift: 10.073735237121582
BN ADAPTATION: OFF
🔥 Few-shot + FT acc: 0.41428571939468384

SUBJECT TEST: subject_04
Shift: 12.814555168151855
BN ADAPTATION: OFF
🔥 Few-shot + FT acc: 0.4619047939777374

SUBJECT TEST: subject_05
Shift: 12.219395637512207
BN ADAPTATION: OFF
🔥 Few-shot + FT acc: 0.380952388048172

SUBJECT TEST: subject_06
Shift: 10.622779846191406
BN ADAPTATION: OFF
🔥 Few-shot + FT acc: 0.4000000059604645

SUBJECT TEST: subject_07
Shift: 9.680804252624512
BN ADAPTATION: OFF
🔥 Few-shot + FT acc: 0.41904762387275696

SUBJECT TEST: subject_08
Shift: 9.870081901550293
BN ADAPTATION: OFF
🔥 Few-shot + FT acc: 0.4285714328289032

SUBJECT TEST: subject_09
Shift: 10.077982902526855
BN ADAPTATION: OFF
🔥 Few-shot + FT acc: 0.37142857909202576

S

In [ ]:
print("e.requires_grad:", e.requires_grad)
print("logits.requires_grad:", logits.requires_grad)
print("loss.requires_grad:", loss.requires_grad)

e.requires_grad: False
logits.requires_grad: False
loss.requires_grad: False


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

print("\n==============================")
print("STARTING PER-SUBJECT EVAL (FIXED)")
print("==============================")

unique_subjects = np.unique(subjects)

results_zs = []
results_fs = []

print("\nTotal subjects:", len(unique_subjects))
print("Subjects:", unique_subjects)


# -------------------------
# FIXED EMBEDDING FUNCTION
# -------------------------
def extract_embeddings_fixed(model, loader, device):
    model.eval()
    E, Y = [], []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)

            # 🔥 FIX: NO db passed
            e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            E.append(e.cpu())
            Y.append(yb)

    return torch.cat(E), torch.cat(Y)


# -------------------------
# MAIN LOOP
# -------------------------
for subj in unique_subjects:

    print("\n" + "="*70)
    print("SUBJECT:", subj)
    print("="*70)

    # -------------------------
    # SELECT SUBJECT DATA
    # -------------------------
    mask = (subjects == subj)

    X_sub   = X_time_t[mask]
    bp_sub  = X_bp_t[mask]
    cov_sub = X_cov_t[mask]
    plv_sub = X_plv_t[mask]
    y_sub   = y_t[mask]

    print("Num windows:", len(y_sub))

    unique, counts = np.unique(y_sub.numpy(), return_counts=True)
    print("Label distribution:", dict(zip(unique, counts)))

    # -------------------------
    # DATASET (🔥 ROI FIXED → size 0)
    # -------------------------
    ds = TensorDataset(
        X_sub,
        bp_sub,
        cov_sub,
        plv_sub,
        torch.zeros(len(y_sub), 0),  # 🔥 FIX: NO ROI
        y_sub,
        torch.zeros(len(y_sub)),     # dummy subject
        torch.ones(len(y_sub))       # dataset ID
    )

    loader = DataLoader(ds, batch_size=256, shuffle=False)
    print("Num batches:", len(loader))

    # -------------------------
    # EMBEDDINGS
    # -------------------------
    E, Y = extract_embeddings_fixed(model, loader, device)

    print("Embedding shape:", E.shape)

    E = F.normalize(E, dim=1)

    # -------------------------
    # ZERO-SHOT
    # -------------------------
    classes = torch.unique(Y)
    print("Classes:", classes.tolist())

    protos = torch.stack([
        E[Y == c].mean(dim=0)
        for c in classes
    ])
    protos = F.normalize(protos, dim=1)

    logits = E @ protos.T
    preds = logits.argmax(dim=1)

    label_map = {int(c.item()): i for i, c in enumerate(classes)}
    Y_local = torch.tensor([label_map[int(v.item())] for v in Y])

    acc_zs = (preds == Y_local).float().mean().item()
    print("🔥 Zero-shot acc:", acc_zs)

    results_zs.append(acc_zs)

    pred_unique, pred_counts = np.unique(preds.numpy(), return_counts=True)
    print("Pred distribution:", dict(zip(pred_unique, pred_counts)))

    # -------------------------
    # FEW-SHOT
    # -------------------------
    FEW_SHOT_PER_CLASS = 20

    support_idx = []

    for c in classes:
        class_idx = (Y == c).nonzero(as_tuple=True)[0]

        if len(class_idx) < FEW_SHOT_PER_CLASS:
            print(f"WARNING: class {c} has only {len(class_idx)} samples")

        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])

    support_idx = torch.cat(support_idx)

    mask_fs = torch.zeros(len(Y), dtype=torch.bool)
    mask_fs[support_idx] = True

    e_support = E[mask_fs]
    y_support = Y[mask_fs]

    e_query   = E[~mask_fs]
    y_query   = Y[~mask_fs]

    print("\nFew-shot split:")
    print("  Support:", len(e_support))
    print("  Query  :", len(e_query))

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    protos_fs = torch.stack([
        e_support[y_support == c].mean(dim=0)
        for c in classes
    ])
    protos_fs = F.normalize(protos_fs, dim=1)

    logits_fs = e_query @ protos_fs.T
    preds_fs = logits_fs.argmax(dim=1)

    y_query_local = torch.tensor([
        label_map[int(v.item())] for v in y_query
    ])

    acc_fs = (preds_fs == y_query_local).float().mean().item()
    print("🔥 Few-shot acc:", acc_fs)

    results_fs.append(acc_fs)


# -------------------------
# FINAL SUMMARY
# -------------------------
print("\n" + "="*70)
print("FINAL RESULTS")
print("="*70)

print("Zero-shot per subject:", results_zs)
print("Few-shot per subject :", results_fs)

print("\nZero-shot mean:", np.mean(results_zs))
print("Zero-shot std :", np.std(results_zs))

print("\nFew-shot mean:", np.mean(results_fs))
print("Few-shot std :", np.std(results_fs))

print("\nBest subject (FS):", np.max(results_fs))
print("Worst subject (FS):", np.min(results_fs))


STARTING PER-SUBJECT EVAL (FIXED)

Total subjects: 10
Subjects: ['subject_01' 'subject_02' 'subject_03' 'subject_04' 'subject_05'
 'subject_06' 'subject_07' 'subject_08' 'subject_09' 'subject_10']

SUBJECT: subject_01
Num windows: 270
Label distribution: {np.int64(0): np.int64(90), np.int64(1): np.int64(90), np.int64(2): np.int64(90)}
Num batches: 2
Embedding shape: torch.Size([270, 64])
Classes: [0, 1, 2]
🔥 Zero-shot acc: 0.5074074268341064
Pred distribution: {np.int64(0): np.int64(64), np.int64(1): np.int64(121), np.int64(2): np.int64(85)}

Few-shot split:
  Support: 60
  Query  : 210
🔥 Few-shot acc: 0.3619047701358795

SUBJECT: subject_02
Num windows: 270
Label distribution: {np.int64(0): np.int64(90), np.int64(1): np.int64(90), np.int64(2): np.int64(90)}
Num batches: 2
Embedding shape: torch.Size([270, 64])
Classes: [0, 1, 2]
🔥 Zero-shot acc: 0.6740740537643433
Pred distribution: {np.int64(0): np.int64(92), np.int64(1): np.int64(93), np.int64(2): np.int64(85)}

Few-shot split:
  S

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

print("\n==============================")
print("STARTING PER-SUBJECT EVAL (FINAL FIXED)")
print("==============================")

unique_subjects = np.unique(subjects)

results_zs = []
results_fs = []

print("\nTotal subjects:", len(unique_subjects))
print("Subjects:", unique_subjects)


# -------------------------
# FIXED EMBEDDING FUNCTION
# -------------------------
def extract_embeddings_fixed(model, loader, device):
    model.eval()
    E, Y = [], []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            E.append(e.cpu())
            Y.append(yb)

    return torch.cat(E), torch.cat(Y)


# -------------------------
# MAIN LOOP
# -------------------------
for subj in unique_subjects:

    print("\n" + "="*70)
    print("SUBJECT:", subj)
    print("="*70)

    # -------------------------
    # SELECT SUBJECT DATA
    # -------------------------
    mask = (subjects == subj)

    X_sub   = X_time_t[mask]
    bp_sub  = X_bp_t[mask]
    cov_sub = X_cov_t[mask]
    plv_sub = X_plv_t[mask]
    y_sub   = y_t[mask]

    print("Num windows:", len(y_sub))

    unique, counts = np.unique(y_sub.numpy(), return_counts=True)
    print("Label distribution:", dict(zip(unique, counts)))

    # -------------------------
    # DATASET (NO ROI)
    # -------------------------
    ds = TensorDataset(
        X_sub,
        bp_sub,
        cov_sub,
        plv_sub,
        torch.zeros(len(y_sub), 0),  # ✅ FIXED
        y_sub,
        torch.zeros(len(y_sub)),     # dummy subject
        torch.ones(len(y_sub))       # dataset ID
    )

    loader = DataLoader(ds, batch_size=256, shuffle=False)
    print("Num batches:", len(loader))

    # -------------------------
    # EMBEDDINGS
    # -------------------------
    E, Y = extract_embeddings_fixed(model, loader, device)

    print("Embedding shape:", E.shape)

    E = F.normalize(E, dim=1)

    # -------------------------
    # ZERO-SHOT
    # -------------------------
    classes = torch.unique(Y)
    print("Classes:", classes.tolist())

    protos = torch.stack([
        E[Y == c].mean(dim=0)
        for c in classes
    ])
    protos = F.normalize(protos, dim=1)

    logits = E @ protos.T
    preds = logits.argmax(dim=1)

    label_map = {int(c.item()): i for i, c in enumerate(classes)}
    Y_local = torch.tensor([label_map[int(v.item())] for v in Y])

    acc_zs = (preds == Y_local).float().mean().item()
    print("🔥 Zero-shot acc:", acc_zs)

    results_zs.append(acc_zs)

    pred_unique, pred_counts = np.unique(preds.numpy(), return_counts=True)
    print("Pred distribution:", dict(zip(pred_unique, pred_counts)))

    # -------------------------
    # FEW-SHOT (🔥 FIXED)
    # -------------------------
    FEW_SHOT_PER_CLASS = 20

    support_idx = []

    for c in torch.unique(Y):
        class_idx = (Y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])

    support_idx = torch.cat(support_idx)

    mask_fs = torch.zeros(len(Y), dtype=torch.bool)
    mask_fs[support_idx] = True

    e_support = E[mask_fs]
    y_support = Y[mask_fs]

    e_query   = E[~mask_fs]
    y_query   = Y[~mask_fs]

    print("\nFew-shot split:")
    print("  Support:", len(e_support))
    print("  Query  :", len(e_query))

    # normalize
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # 🔥 FIX: use support classes only
    classes_fs = torch.unique(y_support)

    label_map_fs = {int(c.item()): i for i, c in enumerate(classes_fs)}

    y_support_mapped = torch.tensor([label_map_fs[int(v.item())] for v in y_support])
    y_query_mapped   = torch.tensor([label_map_fs[int(v.item())] for v in y_query])

    protos_fs = torch.stack([
        e_support[y_support_mapped == i].mean(dim=0)
        for i in range(len(classes_fs))
    ])

    protos_fs = F.normalize(protos_fs, dim=1)

    logits_fs = e_query @ protos_fs.T
    preds_fs = logits_fs.argmax(dim=1)

    acc_fs = (preds_fs == y_query_mapped).float().mean().item()
    print("🔥 Few-shot acc:", acc_fs)

    results_fs.append(acc_fs)


# -------------------------
# FINAL SUMMARY
# -------------------------
print("\n" + "="*70)
print("FINAL RESULTS")
print("="*70)

print("Zero-shot per subject:", results_zs)
print("Few-shot per subject :", results_fs)

print("\nZero-shot mean:", np.mean(results_zs))
print("Zero-shot std :", np.std(results_zs))

print("\nFew-shot mean:", np.mean(results_fs))
print("Few-shot std :", np.std(results_fs))

print("\nBest subject (FS):", np.max(results_fs))
print("Worst subject (FS):", np.min(results_fs))


STARTING PER-SUBJECT EVAL (FINAL FIXED)

Total subjects: 10
Subjects: ['subject_01' 'subject_02' 'subject_03' 'subject_04' 'subject_05'
 'subject_06' 'subject_07' 'subject_08' 'subject_09' 'subject_10']

SUBJECT: subject_01
Num windows: 270
Label distribution: {np.int64(0): np.int64(90), np.int64(1): np.int64(90), np.int64(2): np.int64(90)}
Num batches: 2
Embedding shape: torch.Size([270, 64])
Classes: [0, 1, 2]
🔥 Zero-shot acc: 0.5074074268341064
Pred distribution: {np.int64(0): np.int64(64), np.int64(1): np.int64(121), np.int64(2): np.int64(85)}

Few-shot split:
  Support: 60
  Query  : 210
🔥 Few-shot acc: 0.44285714626312256

SUBJECT: subject_02
Num windows: 270
Label distribution: {np.int64(0): np.int64(90), np.int64(1): np.int64(90), np.int64(2): np.int64(90)}
Num batches: 2
Embedding shape: torch.Size([270, 64])
Classes: [0, 1, 2]
🔥 Zero-shot acc: 0.6740740537643433
Pred distribution: {np.int64(0): np.int64(92), np.int64(1): np.int64(93), np.int64(2): np.int64(85)}

Few-shot spl

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4
SHIFT_THRESH = 0.15

unique_subjects = np.unique(subjects)

all_results = []

print("\n==============================")
print("LOSO EVAL (MATCHED TO OLD PIPELINE)")
print("==============================")

for fold_i, test_subj in enumerate(unique_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1} | TEST SUBJECT: {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask = (subjects == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP (IDENTICAL)
    # =========================
    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    X_cov_train = X_cov_t[train_idx].clone()
    X_cov_test  = X_cov_t[test_idx].clone()

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    # 🔥 SAME NORMALIZATION AS BEFORE
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6

    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6

    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # ROI = NONE (IMPORTANT)
    X_roi_train = torch.zeros((len(train_idx), 0))
    X_roi_test  = torch.zeros((len(test_idx), 0))

    # =========================
    # DATASETS
    # =========================
    train_ds = TensorDataset(
        X_time_train, X_bp_train, X_cov_train, X_plv_train,
        X_roi_train, y_train,
        torch.zeros(len(y_train)),
        torch.ones(len(y_train))
    )

    test_ds = TensorDataset(
        X_time_test, X_bp_test, X_cov_test, X_plv_test,
        X_roi_test, y_test,
        torch.zeros(len(y_test)),
        torch.ones(len(y_test))
    )

    train_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # EMBEDDINGS
    # =========================
    e_train, y_train_full = extract_embeddings_fixed(model, train_loader, device)
    e_test,  y_test_full  = extract_embeddings_fixed(model, test_loader, device)

    # =========================
    # SHIFT DETECTION
    # =========================
    shift = torch.norm(e_train.mean(0) - e_test.mean(0)).item()
    print("Shift:", shift)

    # =========================
    # FEW-SHOT SPLIT
    # =========================
    support_idx = []
    for c in torch.unique(y_test_full):
        idx = (y_test_full == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(idx))
        support_idx.append(idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(y_test_full), dtype=torch.bool)
    mask[support_idx] = True

    # =========================
    # BN ADAPT (GATED)
    # =========================
    if shift > SHIFT_THRESH:
        print("BN ADAPTATION: ON")

        set_partial_bn_adapt(model)

        support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
        support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, *_ in support_loader:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # 🔥 re-extract embeddings
        e_test, y_test_full = extract_embeddings_fixed(model, test_loader, device)

    else:
        print("BN ADAPTATION: OFF")

    # =========================
    # FEW-SHOT CLASSIFIER (IDENTICAL)
    # =========================
    e_support = e_test[mask]
    y_support = y_test_full[mask]

    e_query   = e_test[~mask]
    y_query   = y_test_full[~mask]

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # 🔥 FEATURE WEIGHTING
    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support *= weights
    e_query   *= weights

    classes = torch.unique(y_support)

    protos = torch.stack([
        e_support[y_support == c].mean(dim=0)
        for c in classes
    ])

    # 🔥 MAHALANOBIS
    residuals = torch.cat([
        e_support[y_support == c] - protos[i]
        for i, c in enumerate(classes)
    ])

    D = residuals.shape[1]
    cov = (residuals.T @ residuals) / max(len(residuals)-1,1)

    avg_var = torch.trace(cov) / D
    cov = (1 - SHRINK)*cov + SHRINK*(avg_var*torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    preds = dists.argmin(dim=1)

    acc = (preds == y_query).float().mean().item()

    print("🔥 FEW-SHOT ACC:", acc)

    all_results.append(acc)


print("\n==============================")
print("FINAL LOSO RESULTS")
print("==============================")

print("Mean:", np.mean(all_results))
print("Std :", np.std(all_results))


LOSO EVAL (MATCHED TO OLD PIPELINE)

FOLD 1 | TEST SUBJECT: subject_01
Shift: 0.597216010093689
BN ADAPTATION: ON


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [ ]:
from torch.utils.data import Sampler
import random
from collections import defaultdict

class SubjectBalancedBatchSampler(Sampler):
    """
    Creates batches containing samples from multiple subjects.
    Works with your existing EEGDataset (which has .subj tensor).
    """

    def __init__(self, dataset, batch_size, subjects_per_batch=4):
        self.dataset = dataset
        self.batch_size = batch_size
        self.subjects_per_batch = subjects_per_batch

        # Build index list per subject
        self.subj_to_indices = defaultdict(list)
        for idx, s in enumerate(dataset.subj.tolist()):
            self.subj_to_indices[s].append(idx)

        self.subjects = list(self.subj_to_indices.keys())

        # samples per subject inside batch
        self.samples_per_subject = batch_size // subjects_per_batch

    def __iter__(self):
        # Shuffle indices per subject
        for s in self.subjects:
            random.shuffle(self.subj_to_indices[s])

        # Create subject iterators
        subj_iters = {s: iter(self.subj_to_indices[s]) for s in self.subjects}

        while True:
            # randomly choose subjects for this batch
            chosen_subjects = random.sample(self.subjects, self.subjects_per_batch)

            batch = []

            for s in chosen_subjects:
                for _ in range(self.samples_per_subject):
                    try:
                        batch.append(next(subj_iters[s]))
                    except StopIteration:
                        return  # stop when any subject runs out

            if len(batch) == self.batch_size:
                yield batch
            else:
                return

    def __len__(self):
        # approximate number of batches
        min_samples = min(len(v) for v in self.subj_to_indices.values())
        return (min_samples // self.samples_per_subject)

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4
SHIFT_THRESH = 0.15

unique_subjects = np.unique(subjects)

all_results = []

print("\n==============================")
print("LOSO EVAL (FINAL CLEAN VERSION)")
print("==============================")


# -------------------------
# FIXED EMBEDDING EXTRACTOR (GPU SAFE)
# -------------------------
def extract_embeddings_fixed(model, loader, device):
    model.eval()
    E, Y = [], []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            E.append(e)
            Y.append(yb.to(device))

    return torch.cat(E), torch.cat(Y)


# -------------------------
# MAIN LOSO LOOP
# -------------------------
for fold_i, test_subj in enumerate(unique_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1} | TEST SUBJECT: {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask = (subjects == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    X_cov_train = X_cov_t[train_idx].clone()
    X_cov_test  = X_cov_t[test_idx].clone()

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    # -------------------------
    # NORMALIZATION (CRITICAL)
    # -------------------------
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6

    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6

    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # ROI = NONE
    X_roi_train = torch.zeros((len(train_idx), 0))
    X_roi_test  = torch.zeros((len(test_idx), 0))

    # =========================
    # DATASETS
    # =========================
    train_ds = TensorDataset(
        X_time_train, X_bp_train, X_cov_train, X_plv_train,
        X_roi_train, y_train,
        torch.zeros(len(y_train)),
        torch.ones(len(y_train))
    )

    test_ds = TensorDataset(
        X_time_test, X_bp_test, X_cov_test, X_plv_test,
        X_roi_test, y_test,
        torch.zeros(len(y_test)),
        torch.ones(len(y_test))
    )

    train_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # EMBEDDINGS
    # =========================
    e_train, y_train_full = extract_embeddings_fixed(model, train_loader, device)
    e_test,  y_test_full  = extract_embeddings_fixed(model, test_loader, device)

    # =========================
    # SHIFT DETECTION
    # =========================
    shift = torch.norm(e_train.mean(0) - e_test.mean(0)).item()
    print("Shift:", shift)

    # =========================
    # FEW-SHOT SPLIT
    # =========================
    support_idx = []
    for c in torch.unique(y_test_full):
        idx = (y_test_full == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(idx))
        support_idx.append(idx[:FEW_SHOT_PER_CLASS])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(y_test_full), dtype=torch.bool, device=device)
    mask[support_idx] = True

    # =========================
    # BN ADAPTATION (GATED)
    # =========================
    if shift > SHIFT_THRESH:
        print("BN ADAPTATION: ON")

        set_partial_bn_adapt(model)

        support_subset = torch.utils.data.Subset(test_ds, support_idx.cpu().tolist())
        support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, *_ in support_loader:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        # 🔥 re-extract embeddings after adaptation
        e_test, y_test_full = extract_embeddings_fixed(model, test_loader, device)

    else:
        print("BN ADAPTATION: OFF")

    # =========================
    # FEW-SHOT CLASSIFIER
    # =========================
    e_support = e_test[mask]
    y_support = y_test_full[mask]

    e_query   = e_test[~mask]
    y_query   = y_test_full[~mask]

    # normalize
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # -------------------------
    # FEATURE WEIGHTING
    # -------------------------
    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    # -------------------------
    # PROTOTYPES
    # -------------------------
    classes = torch.unique(y_support)

    protos = torch.stack([
        e_support[y_support == c].mean(dim=0)
        for c in classes
    ])

    # -------------------------
    # MAHALANOBIS (FULL FIX)
    # -------------------------
    residuals = torch.cat([
        e_support[y_support == c] - protos[i]
        for i, c in enumerate(classes)
    ], dim=0)

    D = residuals.shape[1]

    cov = (residuals.T @ residuals) / max(len(residuals)-1, 1)

    avg_var = torch.trace(cov) / D
    eye = torch.eye(D, device=device)

    cov = (1 - SHRINK) * cov + SHRINK * (avg_var * eye)

    cov_inv = torch.linalg.pinv(cov)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    preds = dists.argmin(dim=1)

    acc = (preds == y_query).float().mean().item()

    print("🔥 FEW-SHOT ACC:", acc)

    all_results.append(acc)


print("\n==============================")
print("FINAL LOSO RESULTS")
print("==============================")

print("Mean:", np.mean(all_results))
print("Std :", np.std(all_results))


LOSO EVAL (FINAL CLEAN VERSION)

FOLD 1 | TEST SUBJECT: subject_01
Shift: 19.67873764038086
BN ADAPTATION: ON
🔥 FEW-SHOT ACC: 0.5142857432365417

FOLD 2 | TEST SUBJECT: subject_02
Shift: 23.744749069213867
BN ADAPTATION: ON
🔥 FEW-SHOT ACC: 0.6095238327980042

FOLD 3 | TEST SUBJECT: subject_03
Shift: 15.220579147338867
BN ADAPTATION: ON
🔥 FEW-SHOT ACC: 0.41904762387275696

FOLD 4 | TEST SUBJECT: subject_04
Shift: 54.9047737121582
BN ADAPTATION: ON
🔥 FEW-SHOT ACC: 0.29523810744285583

FOLD 5 | TEST SUBJECT: subject_05
Shift: 32.1258544921875
BN ADAPTATION: ON
🔥 FEW-SHOT ACC: 0.4047619104385376

FOLD 6 | TEST SUBJECT: subject_06
Shift: 14.586030006408691
BN ADAPTATION: ON
🔥 FEW-SHOT ACC: 0.39047619700431824

FOLD 7 | TEST SUBJECT: subject_07
Shift: 37.01194763183594
BN ADAPTATION: ON
🔥 FEW-SHOT ACC: 0.4333333373069763

FOLD 8 | TEST SUBJECT: subject_08
Shift: 32.896121978759766
BN ADAPTATION: ON
🔥 FEW-SHOT ACC: 0.380952388048172

FOLD 9 | TEST SUBJECT: subject_09
Shift: 30.77589797973632

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4
SHIFT_THRESH = 0.15

unique_subjects = np.unique(subjects)

all_results = []

print("\n==============================")
print("LOSO (STRICT MATCH VERSION)")
print("==============================")


def extract_embeddings_fixed(model, loader, device):
    model.eval()
    E, Y = [], []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            E.append(e.cpu())   # 👈 MATCH OLD CODE (CPU STORAGE)
            Y.append(yb.cpu())

    return torch.cat(E), torch.cat(Y)


for fold_i, test_subj in enumerate(unique_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1} | TEST SUBJECT: {test_subj}")
    print("="*80)

    # =========================
    # SPLIT (IDENTICAL)
    # =========================
    test_mask = (subjects == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURES (CLONE EXACTLY)
    # =========================
    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    X_cov_train = X_cov_t[train_idx].clone()
    X_cov_test  = X_cov_t[test_idx].clone()

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    # =========================
    # NORMALIZATION (IDENTICAL)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6

    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6

    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # ROI EMPTY (same as your old setup)
    X_roi_train = torch.zeros((len(train_idx), 0))
    X_roi_test  = torch.zeros((len(test_idx), 0))

    # =========================
    # DATASETS
    # =========================
    train_ds = TensorDataset(
        X_time_train, X_bp_train, X_cov_train, X_plv_train,
        X_roi_train, y_train,
        torch.zeros(len(y_train)),
        torch.ones(len(y_train))
    )

    test_ds = TensorDataset(
        X_time_test, X_bp_test, X_cov_test, X_plv_test,
        X_roi_test, y_test,
        torch.zeros(len(y_test)),
        torch.ones(len(y_test))
    )

    train_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # EMBEDDINGS (CPU!)
    # =========================
    e_train, y_train_full = extract_embeddings_fixed(model, train_loader, device)
    e_test,  y_test_full  = extract_embeddings_fixed(model, test_loader, device)

    # =========================
    # SHIFT (IDENTICAL)
    # =========================
    shift = torch.norm(e_train.mean(0) - e_test.mean(0)).item()
    print("Shift:", shift)

    # =========================
    # FEW-SHOT SPLIT (FIXED CORRECTLY)
    # =========================
    torch.manual_seed(1234 + fold_i)  # 👈 SAME AS OLD CODE
    np.random.seed(1234 + fold_i)

    support_idx = []

    for c in torch.unique(y_test_full):
        class_idx = (y_test_full == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])

    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(y_test_full), dtype=torch.bool)
    mask[support_idx] = True

    # =========================
    # BN ADAPTATION (IDENTICAL)
    # =========================
    if shift > SHIFT_THRESH:
        print("BN ADAPTATION: ON")

        set_partial_bn_adapt(model)

        support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
        support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

        with torch.no_grad():
            for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, *_ in support_loader:
                xb_time = xb_time.to(device)
                xb_bp   = xb_bp.to(device)
                xb_cov  = xb_cov.to(device)
                xb_plv  = xb_plv.to(device)
                xb_roi  = xb_roi.to(device)
                _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

        e_test, y_test_full = extract_embeddings_fixed(model, test_loader, device)

    else:
        print("BN ADAPTATION: OFF")

    # =========================
    # FEW-SHOT CLASSIFIER (CPU!)
    # =========================
    e_support = e_test[mask]
    y_support = y_test_full[mask]

    e_query   = e_test[~mask]
    y_query   = y_test_full[~mask]

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # FEATURE WEIGHTING
    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)

    protos = torch.stack([
        e_support[y_support == c].mean(dim=0)
        for c in classes
    ])

    # MAHALANOBIS (CPU SAFE)
    residuals = torch.cat([
        e_support[y_support == c] - protos[i]
        for i, c in enumerate(classes)
    ])

    D = residuals.shape[1]

    cov = (residuals.T @ residuals) / max(len(residuals)-1, 1)

    avg_var = torch.trace(cov) / D
    cov = (1 - SHRINK)*cov + SHRINK*(avg_var * torch.eye(D))

    cov_inv = torch.linalg.pinv(cov)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    preds = dists.argmin(dim=1)

    acc = (preds == y_query).float().mean().item()

    print("🔥 FEW-SHOT ACC:", acc)

    all_results.append(acc)


print("\n==============================")
print("FINAL RESULTS")
print("==============================")

print("Mean:", np.mean(all_results))
print("Std :", np.std(all_results))


LOSO (STRICT MATCH VERSION)

FOLD 1 | TEST SUBJECT: subject_01
Shift: 19.690011978149414
BN ADAPTATION: ON
🔥 FEW-SHOT ACC: 0.4476190507411957

FOLD 2 | TEST SUBJECT: subject_02
Shift: 23.812217712402344
BN ADAPTATION: ON
🔥 FEW-SHOT ACC: 0.538095235824585

FOLD 3 | TEST SUBJECT: subject_03
Shift: 15.221043586730957
BN ADAPTATION: ON
🔥 FEW-SHOT ACC: 0.4523809552192688

FOLD 4 | TEST SUBJECT: subject_04
Shift: 54.862518310546875
BN ADAPTATION: ON
🔥 FEW-SHOT ACC: 0.44285714626312256

FOLD 5 | TEST SUBJECT: subject_05
Shift: 32.1608772277832
BN ADAPTATION: ON
🔥 FEW-SHOT ACC: 0.41904762387275696

FOLD 6 | TEST SUBJECT: subject_06
Shift: 14.583154678344727
BN ADAPTATION: ON
🔥 FEW-SHOT ACC: 0.4095238149166107

FOLD 7 | TEST SUBJECT: subject_07
Shift: 37.025474548339844
BN ADAPTATION: ON
🔥 FEW-SHOT ACC: 0.3857142925262451

FOLD 8 | TEST SUBJECT: subject_08
Shift: 32.960235595703125
BN ADAPTATION: ON
🔥 FEW-SHOT ACC: 0.5095238089561462

FOLD 9 | TEST SUBJECT: subject_09
Shift: 30.84697151184082


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

print("\n==============================")
print("STARTING PER-SUBJECT EVALUATION")
print("==============================")

unique_subjects = np.unique(subjects)
results = []

print("\nTotal subjects:", len(unique_subjects))
print("Subjects list:", unique_subjects)

for subj in unique_subjects:

    print("\n" + "="*60)
    print("SUBJECT:", subj)
    print("="*60)

    # -------------------------
    # MASK
    # -------------------------
    mask = (subjects == subj)
    idx = np.where(mask)[0]

    print("Num windows:", len(idx))

    # -------------------------
    # SLICE DATA
    # -------------------------
    X_sub  = X_time_t[mask]
    bp_sub = X_bp_t[mask]
    cov_sub= X_cov_t[mask]
    plv_sub= X_plv_t[mask]
    y_sub  = y_t[mask]

    print("Shapes:")
    print("  X_time :", X_sub.shape)
    print("  X_bp   :", bp_sub.shape)
    print("  X_cov  :", cov_sub.shape)
    print("  X_plv  :", plv_sub.shape)
    print("  y      :", y_sub.shape)

    print("Label distribution:")
    unique, counts = np.unique(y_sub.numpy(), return_counts=True)
    print(dict(zip(unique, counts)))

    # -------------------------
    # BUILD DATASET
    # -------------------------
    ds = TensorDataset(
        X_sub,
        bp_sub,
        cov_sub,
        plv_sub,
        torch.zeros(len(y_sub), 3),   # ROI
        y_sub,
        torch.zeros(len(y_sub)),      # dummy subject
        torch.ones(len(y_sub))        # dataset ID
    )

    loader = DataLoader(ds, batch_size=256, shuffle=False)

    print("Loader batches:", len(loader))

    # -------------------------
    # EXTRACT EMBEDDINGS
    # -------------------------
    model.eval()
    with torch.no_grad():
        E, Y = extract_embeddings(model, loader, device)

    print("Embedding shape:", E.shape)
    print("Labels shape:", Y.shape)

    # -------------------------
    # NORMALIZE
    # -------------------------
    E = F.normalize(E, dim=1)

    print("Embedding stats:")
    print("  mean:", E.mean().item())
    print("  std :", E.std().item())

    # -------------------------
    # PROTOTYPES
    # -------------------------
    classes = torch.unique(Y)
    print("Classes in subject:", classes.tolist())

    protos = torch.stack([
        E[Y == c].mean(dim=0)
        for c in classes
    ])

    protos = F.normalize(protos, dim=1)

    print("Prototypes shape:", protos.shape)

    # -------------------------
    # CLASSIFICATION
    # -------------------------
    logits = E @ protos.T
    preds = logits.argmax(dim=1)

    # remap labels
    label_map = {int(c.item()): i for i, c in enumerate(classes)}
    Y_local = torch.tensor([label_map[int(v.item())] for v in Y])

    acc = (preds == Y_local).float().mean().item()

    print("Zero-shot accuracy:", acc)

    # -------------------------
    # EXTRA DEBUG
    # -------------------------
    print("Prediction distribution:")
    pred_unique, pred_counts = np.unique(preds.numpy(), return_counts=True)
    print(dict(zip(pred_unique, pred_counts)))

    results.append(acc)

# -------------------------
# FINAL SUMMARY
# -------------------------
print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)

print("All accuracies:", results)
print("Mean accuracy:", np.mean(results))
print("Std accuracy :", np.std(results))
print("Best subject :", np.max(results))
print("Worst subject:", np.min(results))